In [1]:
import pandas as pd

#FACT: Call Activity Data Set - Dials(Call Activity Data Set 2026-02-).csv
calls = pd.read_csv('Call Activity Data Set - Dials(Call Activity Data Set 2026-02-).csv')

# Filter to direction = 'outbound'
calls = calls[calls['Direction'] == 'Outbound']

# Remove records where lead create (et) time > activity (et) time
calls = calls[
    calls['Lead Created (ET) Time'] <= calls['Activity (ET) Time']
]


#DIM: Lead(Sheet1).csv
leads = pd.read_csv(
    "DTC Leads Data Set - dim lead(Sheet1).csv",
    parse_dates=[
        'Lead Created (ET) Time',
        'Last Contact Attempt (ET) Time'
    ],
)

leads = leads[leads['Last Contact Attempt (ET) Time'].notna()]

# CONVERT cols to boolean
yes_no_columns = [
    'Contacted? (Yes / No)',
    'Is Live Transfer? (Yes / No)',
    'Is Latest Pool A Dial Attempt? (Yes / No)',
    'Is Latest Dial Attempt? (Yes / No)',
    'Is Pool A (Active SDR)? (Yes / No)',
    'Incoming Lead? (Yes / No)'
]

# Convert "Yes" → True, "No" → False (safe mapping)
for col in yes_no_columns:
    if col in calls.columns:
        calls[col] = calls[col].map({
            'Yes': True,
            'No': False
        }).astype('boolean')  # pandas nullable boolean type (keeps NaN)
    else:
        print(f"Column not found: {col}")

# Quick verification
print("\nPost-conversion summary:")
for col in yes_no_columns:
    if col in calls.columns:
        print(f"\n{col}:")
        print(calls[col].value_counts(dropna=False))
        print(f"  dtype: {calls[col].dtype}")


Post-conversion summary:

Contacted? (Yes / No):
Contacted? (Yes / No)
False    273423
True      15685
Name: count, dtype: Int64
  dtype: boolean

Is Live Transfer? (Yes / No):
Is Live Transfer? (Yes / No)
False    284787
True       4321
Name: count, dtype: Int64
  dtype: boolean

Is Latest Pool A Dial Attempt? (Yes / No):
Is Latest Pool A Dial Attempt? (Yes / No)
False    269463
True      19645
Name: count, dtype: Int64
  dtype: boolean

Is Latest Dial Attempt? (Yes / No):
Is Latest Dial Attempt? (Yes / No)
False    271054
True      18054
Name: count, dtype: Int64
  dtype: boolean

Is Pool A (Active SDR)? (Yes / No):
Is Pool A (Active SDR)? (Yes / No)
True     242546
False     46562
Name: count, dtype: Int64
  dtype: boolean

Incoming Lead? (Yes / No):
Incoming Lead? (Yes / No)
False    289108
Name: count, dtype: Int64
  dtype: boolean


In [2]:
print("=== Leads DataFrame Overview ===")
print("Shape:", leads.shape)
print("\nData types:\n", leads.dtypes)
print("\nMissing values:\n", leads.isnull().sum())
print("\nSample head(5):\n", leads.head().to_string(index=False))

print("\nUnique Current Lead Status:", leads['Current Lead Status'].value_counts(dropna=False).to_dict())
print("Unique Lead Channel Segment:", leads['Lead Channel Segment'].value_counts(dropna=False).to_dict())
print("Unique Borrower Requested Loan Type:", leads['Borrower Requested Loan Type'].value_counts(dropna=False).to_dict())

print("\nFunded loans count:", leads['Number of Funded Loan Leads'].sum())
print("Pre-Approval count:", leads['Number of Pre-Approval Leads'].sum())
print("File Started count:", leads['Number of File Started Leads'].sum())

# ────────────────────────────────────────────────
print("\n\n=== Calls DataFrame Overview (Outbound only) ===")
print("Shape:", calls.shape)
print("\nData types:\n", calls.dtypes)
print("\nMissing values:\n", calls.isnull().sum())
print("\nSample head(3):\n", calls.head(3).to_string(index=False))

print("\nUnique Outcomes:", calls['Outcome'].value_counts(dropna=False).head(10).to_dict())
print("Contacted? distribution:\n", calls['Contacted? (Yes / No)'].value_counts(dropna=False).to_dict())

print("\nCall Attempt Number stats:\n", calls['Call Attempt Number'].describe())

print("\nActivity (ET) Time range:", 
      calls['Activity (ET) Time'].min(), "→", calls['Activity (ET) Time'].max())

print("Lead Created (ET) Time range:", 
      calls['Lead Created (ET) Time'].min(), "→", calls['Lead Created (ET) Time'].max())

# Quick joinability check
print("\nUnique Lead IDs — leads:", leads['Lead ID'].nunique())
print("Unique Lead IDs — calls:", calls['Lead ID'].nunique())
overlap = len(set(leads['Lead ID']) & set(calls['Lead ID']))
print("Overlap (leads present in both):", overlap)

=== Leads DataFrame Overview ===
Shape: (31016, 19)

Data types:
 Lead ID                                          str
Last Contact Attempt (ET) Time        datetime64[us]
Utm Source                                       str
Utm Medium                                       str
Lead Source                                      str
Lead Channel Segment                             str
Borrower Requested Loan Type                     str
Current Lead Status                              str
Lead Created (ET) Date                           str
Lead Created (ET) Time                datetime64[us]
Gross Leads                                    int64
Number of Called Leads                         int64
Number of Contacted Leads                      int64
Number of Opportunities                        int64
Number of File Started Leads                   int64
Number of Entered Processing Leads             int64
Number of Locked Loan Leads                    int64
Number of Pre-Approval Leads     


Sample head(5):
            Lead ID Last Contact Attempt (ET) Time        Utm Source Utm Medium                 Lead Source Lead Channel Segment Borrower Requested Loan Type   Current Lead Status Lead Created (ET) Date Lead Created (ET) Time  Gross Leads  Number of Called Leads  Number of Contacted Leads  Number of Opportunities  Number of File Started Leads  Number of Entered Processing Leads  Number of Locked Loan Leads  Number of Pre-Approval Leads  Number of Funded Loan Leads
00QPh00000ii4plMAA            2026-02-13 03:42:00            google        cpc GuidedExperience - Purchase           Paid Spend                     Purchase          Pre-Approved              1/24/2026    2026-01-24 15:28:00            1                       1                          1                        1                             1                                   0                            0                             1                            0
00QPh00000iFPzFMAW            2026-02-13 03:28

In [3]:
# Filter to Pool A calls only (non-null attempt number)
pool_a = calls[calls['Pool A Call Attempt Number'].notna()].copy()

# Convert attempt to integer for cleaner columns
pool_a['attempt'] = pool_a['Pool A Call Attempt Number'].astype(int)

# Check unique Outcomes in Pool A calls
print("Unique Outcomes in Pool A calls:")
print(pool_a['Outcome'].value_counts(dropna=False))

print("\nNumber of rows in Pool A subset:", len(pool_a))
print("Number of unique leads in Pool A subset:", pool_a['Lead ID'].nunique())

# Optional: top attempt numbers in this subset
print("\nTop Pool A attempt numbers (count of rows):")
print(pool_a['attempt'].value_counts().sort_index().head(25))

Unique Outcomes in Pool A calls:
Outcome
No Answer                                                        223695
Left Voicemail                                                     5237
Hung Up                                                            3762
Transfer Sent                                                      3167
Appointment Set                                                    1649
Nurture - No Benefit                                               1290
Add to DNC                                                          658
Bad/Disconnected Phone                                              611
Redial                                                              595
Nurture - Went With Another Lender                                  586
Hold Time Exceeded                                                  406
Nurture - Credit Score                                              202
Nurture - Income                                                    187
Nurture - Loan Amount U

In [4]:
# How many attempts does the typical lead actually have?
attempt_counts = pool_a.groupby('Lead ID')['attempt'].nunique().value_counts().sort_index()

print("Number of Pool A attempts per lead – distribution:")
print(attempt_counts)

print("\nMedian / mean attempts per lead:")
print(pool_a.groupby('Lead ID')['attempt'].max().describe()[['50%', 'mean']])

print("\nMax attempts observed in data:", pool_a['attempt'].max())

Number of Pool A attempts per lead – distribution:
attempt
1     4678
2     1563
3     1075
4     1009
5      989
6      807
7      597
8      542
9      664
10     480
11     539
12     566
13     732
14    1046
15    1623
16    2175
17    3090
18    1221
19     556
20     626
21      82
22      22
23      15
24       9
25      14
26      10
27      11
28       7
29       9
30       3
31       3
32       1
33       2
35       3
36       2
37       1
39       1
40       3
Name: count, dtype: int64

Median / mean attempts per lead:
50%     12.000000
mean    10.412496
Name: attempt, dtype: float64

Max attempts observed in data: 51


In [5]:
# Using the pool_a we already have (with 'attempt' as int)

# Group by lead + attempt, take the outcome(s) — but since one row per attempt usually, we can take first/min
attempt_outcome = pool_a.groupby(['Lead ID', 'attempt'])['Outcome'].first().reset_index()

# Now pivot: rows = attempt, columns = Outcome, values = count of distinct leads
outcome_by_attempt = pd.crosstab(
    attempt_outcome['attempt'],
    attempt_outcome['Outcome']
)

# Add total leads that reached each attempt
outcome_by_attempt['Total Leads Reached'] = outcome_by_attempt.sum(axis=1)

# Sort columns by frequency (most common outcomes left)
col_order = outcome_by_attempt.sum().sort_values(ascending=False).index
outcome_by_attempt = outcome_by_attempt[col_order]

print("=== Outcome distribution BY attempt number (first outcome seen at that attempt) ===")
print(outcome_by_attempt.head(15))  # first 15 attempts should be most interesting

print("\nAttempts shown:", outcome_by_attempt.index.min(), "to", outcome_by_attempt.index.max())

=== Outcome distribution BY attempt number (first outcome seen at that attempt) ===
Outcome  Total Leads Reached  No Answer  Left Voicemail  Hung Up  \
attempt                                                            
1                      23673      14190            4491      767   
2                      17641      15456             500      479   
3                      16708      15511              59      340   
4                      17182      16198              30      318   
5                      16605      15792              20      282   
6                      15654      14998              18      221   
7                      14783      14272               8      183   
8                      14118      13687              10      161   
9                      13554      13176              17      127   
10                     12894      12553               8      131   
11                     12420      12085              13      126   
12                     11880    

In [6]:
# Ensure flags are boolean/int for clean aggregation
leads['was_contacted'] = leads['Number of Contacted Leads'] >= 1
leads['was_funded']    = leads['Number of Funded Loan Leads'] >= 1

# Group by Lead Channel Segment
summary = leads.groupby('Lead Channel Segment').agg(
    total_leads= ('Lead ID', 'nunique'),
    contacted_leads= ('was_contacted', 'sum'),
    funded_leads= ('was_funded', 'sum')
)

# Calculate rates
summary['fund_rate'] = (summary['funded_leads'] / summary['total_leads']).round(4)
summary['contact_rate'] = (summary['contacted_leads'] / summary['total_leads'] * 1000).round(4)
summary['funded_per_1k_leads'] = (summary['funded_leads'] / summary['total_leads'] * 1000).round(2)
summary['funded_per_1k_contacted']= (summary['funded_leads'] / summary['contacted_leads'] * 1000).round(2).fillna(0)

# Sort by funded per 1k leads descending (most productive first)
summary = summary.sort_values('funded_per_1k_leads', ascending=False)

# Reorder columns for readability
summary = summary[[
    'total_leads',
    'contacted_leads',
    'funded_leads',
    'fund_rate',
    'contact_rate',
    'funded_per_1k_leads',
    'funded_per_1k_contacted'
]]

# Nice display names
summary.columns = [
    '# Leads',
    '# Contacted Leads',
    'Contact Rate',
    '# Funded',
    'Fund Rate',
    'Funded / 1k Leads',
    'Funded / 1k Contacted'
]

print("=== Lead Channel Segment Performance Summary ===")
print(summary.to_string())

=== Lead Channel Segment Performance Summary ===
                      # Leads  # Contacted Leads  Contact Rate  # Funded  Fund Rate  Funded / 1k Leads  Funded / 1k Contacted
Lead Channel Segment                                                                                                         
Retargeting               509                509            14    0.0275  1000.0000              27.50                  27.50
Direct / Organic         3300               1850            84    0.0255   560.6061              25.45                  45.41
Paid Spend               5138               2934            73    0.0142   571.0393              14.21                  24.88
Mail                      200                198             1    0.0050   990.0000               5.00                   5.05
Digital Buy Partner     12509               5386            15    0.0012   430.5700               1.20                   2.78
Movoto                   9236               6638             7    0.0

In [7]:
# Make sure we start fresh with the right data
successful_contacts = calls[
    (calls['Contacted? (Yes / No)'] == True) & 
    (calls['Pool A Call Attempt Number'].notna())
].copy()

# Get the FIRST (earliest) successful attempt per lead
first_contact_attempt = successful_contacts.groupby('Lead ID')['Pool A Call Attempt Number'].min().reset_index(
).rename(columns={'Pool A Call Attempt Number': 'first_success_attempt'})

first_contact_attempt['first_success_attempt'] = first_contact_attempt['first_success_attempt'].astype(int)

# Now the distribution & stats
print("Distribution of attempt number for FIRST successful contact:")
print(first_contact_attempt['first_success_attempt'].value_counts().sort_index().head(20))

print("\nMedian attempt to first contact:", first_contact_attempt['first_success_attempt'].median())
print("Mean attempt to first contact:", first_contact_attempt['first_success_attempt'].mean().round(1))
print("90th percentile:", first_contact_attempt['first_success_attempt'].quantile(0.90))
print("Number of leads ever contacted:", len(first_contact_attempt))

Distribution of attempt number for FIRST successful contact:
first_success_attempt
1     3856
2      978
3      633
4      446
5      360
6      284
7      211
8      173
9      161
10     134
11     117
12     106
13      90
14      64
15      68
16      58
17      47
18      28
19       9
20       5
Name: count, dtype: int64

Median attempt to first contact: 2.0
Mean attempt to first contact: 3.5
90th percentile: 9.0
Number of leads ever contacted: 7840


In [8]:
# Merge the first-contact attempt info with leads (keep only contacted leads for now)
contact_difficulty = first_contact_attempt.merge(
    leads[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='left'
)

# Quick sanity check
print("Shape after merge:", contact_difficulty.shape)
print("\nMissing segments?", contact_difficulty['Lead Channel Segment'].isna().sum())
print("\nSegments present:\n", contact_difficulty['Lead Channel Segment'].value_counts(dropna=False))

Shape after merge: (7840, 3)

Missing segments? 1

Segments present:
 Lead Channel Segment
Movoto                 3427
Digital Buy Partner    2281
Paid Spend             1435
Direct / Organic        645
Other                    43
Retargeting               8
NaN                       1
Name: count, dtype: int64


In [9]:
# Drop the 1 missing segment row
contact_difficulty = contact_difficulty.dropna(subset=['Lead Channel Segment'])

# Aggregate per segment
difficulty_by_segment = contact_difficulty.groupby('Lead Channel Segment').agg(
    contacted_leads_outbound = ('Lead ID', 'nunique'),
    median_attempts          = ('first_success_attempt', 'median'),
    mean_attempts            = ('first_success_attempt', 'mean'),
    p90_attempts             = ('first_success_attempt', lambda x: x.quantile(0.90)),
    pct_attempt_1            = ('first_success_attempt', lambda x: (x == 1).mean() * 100),
    pct_within_3             = ('first_success_attempt', lambda x: (x <= 3).mean() * 100)
).round({'mean_attempts': 1, 'p90_attempts': 1, 'pct_attempt_1': 1, 'pct_within_3': 1})

# Sort by median_attempts ascending (easiest to reach first)
difficulty_by_segment = difficulty_by_segment.sort_values('median_attempts')

print("=== Contact Difficulty by Lead Channel Segment ===")
print("(Among leads that were successfully contacted via outbound Pool A calls)")
print(difficulty_by_segment.to_string())

=== Contact Difficulty by Lead Channel Segment ===
(Among leads that were successfully contacted via outbound Pool A calls)
                      contacted_leads_outbound  median_attempts  mean_attempts  p90_attempts  pct_attempt_1  pct_within_3
Lead Channel Segment                                                                                                     
Direct / Organic                           645              1.0            2.7           7.0           60.6          79.7
Other                                       43              1.0            2.2           5.4           69.8          88.4
Retargeting                                  8              1.0            1.1           1.3           87.5         100.0
Paid Spend                                1435              1.0            2.7           7.0           61.9          78.7
Movoto                                    3427              2.0            3.2           8.0           47.7          70.9
Digital Buy Partner   

In [10]:
# Filter to only rows where outcome = Transfer Sent or Appointment Set
success_calls = pool_a[pool_a['Outcome'].isin(['Transfer Sent', 'Appointment Set'])].copy()

# Add segment (merge from leads)
success_calls = success_calls.merge(
    leads[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='left'
)

# Quick overview
print("Number of Transfer Sent / Appointment Set events in Pool A:", len(success_calls))
print("\nBreakdown by outcome:\n", success_calls['Outcome'].value_counts())

print("\nSegments that have any success events:\n", success_calls['Lead Channel Segment'].value_counts(dropna=False))

print("\nAttempt distribution overall:\n", success_calls['attempt'].value_counts().sort_index().head(15))

Number of Transfer Sent / Appointment Set events in Pool A: 4816

Breakdown by outcome:
 Outcome
Transfer Sent      3167
Appointment Set    1649
Name: count, dtype: int64

Segments that have any success events:
 Lead Channel Segment
Movoto                 1796
Digital Buy Partner    1364
Paid Spend             1112
Direct / Organic        501
Other                    35
Retargeting               6
NaN                       2
Name: count, dtype: int64

Attempt distribution overall:
 attempt
1     2494
2      616
3      375
4      278
5      207
6      161
7      118
8       97
9       87
10      61
11      67
12      55
13      50
14      41
15      32
Name: count, dtype: int64


In [11]:
# Bucket attempts for readability
bins = [0, 1, 2, 3, 6, float('inf')]
labels = ['Attempt 1', 'Attempt 2', 'Attempt 3', 'Attempts 4-6', 'Attempts 7+']
success_calls['attempt_bucket'] = pd.cut(success_calls['attempt'], bins=bins, labels=labels, right=True)

# Aggregate
success_by_segment = success_calls.groupby(['Lead Channel Segment', 'attempt_bucket']).size().unstack(fill_value=0)

# Add totals & percentages
success_by_segment['Total Wins'] = success_by_segment.sum(axis=1)
success_by_segment['% on Attempt 1'] = (success_by_segment.get('Attempt 1', 0) / success_by_segment['Total Wins'] * 100).round(1)
success_by_segment['% within 3'] = (
    (success_by_segment.get('Attempt 1', 0) + 
     success_by_segment.get('Attempt 2', 0) + 
     success_by_segment.get('Attempt 3', 0)) / success_by_segment['Total Wins'] * 100
).round(1)

# Sort by total wins descending
success_by_segment = success_by_segment.sort_values('Total Wins', ascending=False)

# Reorder columns
success_by_segment = success_by_segment[[
    'Total Wins', 'Attempt 1', 'Attempt 2', 'Attempt 3', 'Attempts 4-6', 'Attempts 7+',
    '% on Attempt 1', '% within 3'
]]

print("=== Transfer Sent + Appointment Set by Segment & Attempt Bucket ===")
print(success_by_segment.to_string())

=== Transfer Sent + Appointment Set by Segment & Attempt Bucket ===
attempt_bucket        Total Wins  Attempt 1  Attempt 2  Attempt 3  Attempts 4-6  Attempts 7+  % on Attempt 1  % within 3
Lead Channel Segment                                                                                                    
Movoto                      1796        794        260        178           282          282            44.2        68.6
Digital Buy Partner         1364        685        139         90           182          268            50.2        67.0
Paid Spend                  1112        684        134         76           132           86            61.5        80.4
Direct / Organic             501        305         75         29            48           44            60.9        81.6
Other                         35         21          7          2             2            3            60.0        85.7
Retargeting                    6          5          1          0             0      

In [12]:
# Create attempt buckets up to 15 + tail
def bucket_attempt(n):
    if n <= 15:
        return f'Attempt {int(n)}'
    else:
        return 'Attempt 16+'

success_calls['attempt_bucket'] = success_calls['attempt'].apply(bucket_attempt)

# Aggregate counts per segment and bucket
extended_table = success_calls.groupby(['Lead Channel Segment', 'attempt_bucket']).size().unstack(fill_value=0)

# Add totals & percentages
extended_table['Total Wins'] = extended_table.sum(axis=1)
extended_table['% on Attempt 1'] = (extended_table.get('Attempt 1', 0) / extended_table['Total Wins'] * 100).round(1)
extended_table['% within 3'] = (
    (extended_table.get('Attempt 1', 0) + 
     extended_table.get('Attempt 2', 0) + 
     extended_table.get('Attempt 3', 0)) / extended_table['Total Wins'] * 100
).round(1)

# Reorder columns: totals first, then attempts 1-15, 16+, percentages last
cols = ['Total Wins'] + [f'Attempt {i}' for i in range(1, 16)] + ['Attempt 16+'] + ['% on Attempt 1', '% within 3']
extended_table = extended_table[[c for c in cols if c in extended_table.columns]]

# Sort by Total Wins descending
extended_table = extended_table.sort_values('Total Wins', ascending=False)

print("=== Transfer Sent + Appointment Set by Segment — Attempts 1 to 15 + Tail ===")
print(extended_table.to_string())

=== Transfer Sent + Appointment Set by Segment — Attempts 1 to 15 + Tail ===
attempt_bucket        Total Wins  Attempt 1  Attempt 2  Attempt 3  Attempt 4  Attempt 5  Attempt 6  Attempt 7  Attempt 8  Attempt 9  Attempt 10  Attempt 11  Attempt 12  Attempt 13  Attempt 14  Attempt 15  Attempt 16+  % on Attempt 1  % within 3
Lead Channel Segment                                                                                                                                                                                                                                
Movoto                      1796        794        260        178        124         94         64         45         46         35          23          29          24          19          23          15           23            44.2        68.6
Digital Buy Partner         1364        685        139         90         73         61         48         47         32         31          23          23          22          22         

In [13]:
# Merge opportunity flag from leads
success_with_opp = success_calls.merge(
    leads[['Lead ID', 'Number of Opportunities']],
    on='Lead ID',
    how='left'
)

# Create simple flag: had opportunity or not
success_with_opp['had_opportunity'] = success_with_opp['Number of Opportunities'] >= 1

# Quick overall
print("Total Transfer/Appointment events:", len(success_with_opp))
print("Of which had an Opportunity:", success_with_opp['had_opportunity'].sum())
print("Overall opportunity rate:", 
      (success_with_opp['had_opportunity'].mean() * 100).round(1), "%")

print("\nOpportunity rate by segment:")
print(
    success_with_opp.groupby('Lead Channel Segment')['had_opportunity']
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)

Total Transfer/Appointment events: 4816
Of which had an Opportunity: 3832
Overall opportunity rate: 79.6 %

Opportunity rate by segment:
Lead Channel Segment
Direct / Organic       92.2
Paid Spend             92.2
Digital Buy Partner    85.9
Other                  80.0
Movoto                 63.8
Retargeting            16.7
Name: had_opportunity, dtype: float64


In [14]:
# Using success_with_opp from previous (has had_opportunity flag)

# Flag early vs late first-success
success_with_opp['win_timing'] = 'Attempt 1'
success_with_opp.loc[success_with_opp['attempt'] >= 4, 'win_timing'] = 'Attempt 4+'

# Overall comparison
timing_opp = success_with_opp.groupby('win_timing')['had_opportunity'].agg(['count', 'mean']).round(3)
timing_opp['opp_rate_%'] = timing_opp['mean'] * 100
print("Overall Opportunity Rate by Win Timing:")
print(timing_opp[['count', 'opp_rate_%']])

# By segment
timing_by_segment = success_with_opp.groupby(['Lead Channel Segment', 'win_timing'])['had_opportunity'].mean().unstack().mul(100).round(1)
print("\nOpportunity Rate by Segment & Win Timing (%):")
print(timing_by_segment)

Overall Opportunity Rate by Win Timing:
            count  opp_rate_%
win_timing                   
Attempt 1    3485        83.0
Attempt 4+   1331        70.7

Opportunity Rate by Segment & Win Timing (%):
win_timing            Attempt 1  Attempt 4+
Lead Channel Segment                       
Digital Buy Partner        89.7        78.0
Direct / Organic           93.9        84.8
Movoto                     66.2        58.3
Other                      83.3        60.0
Paid Spend                 94.5        82.6
Retargeting                16.7         NaN


In [15]:
def bin_attempt(n):
    if n == 1:
        return 'Attempt 1'
    elif 1 <= n <= 3:
        return 'Attempts 1–3 (day 1)'
    elif 4 <= n <= 6:
        return 'Attempts 4–6'
    elif 7 <= n <= 9:
        return 'Attempts 7–9'
    elif 10 <= n <= 12:
        return 'Attempts 10–12'
    elif 13 <= n <= 15:
        return 'Attempts 13–15'
    else:
        return '16+'

success_with_opp['attempt_bin'] = success_with_opp['attempt'].apply(bin_attempt)

# Quick check
print("Bin distribution (count of Transfer/Appointment events):")
print(success_with_opp['attempt_bin'].value_counts().sort_index())

Bin distribution (count of Transfer/Appointment events):
attempt_bin
16+                       77
Attempt 1               2494
Attempts 10–12           183
Attempts 13–15           123
Attempts 1–3 (day 1)     991
Attempts 4–6             646
Attempts 7–9             302
Name: count, dtype: int64


In [16]:
# Opportunity rate per bin - overall
opp_by_bin = success_with_opp.groupby('attempt_bin')['had_opportunity'].agg(
    count='size',
    opp_count='sum',
    opp_rate=lambda x: x.mean() * 100
).round({'opp_rate': 1})

opp_by_bin = opp_by_bin[['count', 'opp_count', 'opp_rate']]
opp_by_bin.columns = ['Events', 'Opps', 'Opp Rate %']

print("=== Overall Opportunity Rate by Attempt Bin ===")
print(opp_by_bin.sort_index())

# Per segment (only segments with reasonable volume)
opp_by_segment_bin = success_with_opp.groupby(['Lead Channel Segment', 'attempt_bin'])['had_opportunity'].mean().unstack().mul(100).round(1)

print("\n=== Opportunity Rate (%) by Segment & Attempt Bin ===")
print(opp_by_segment_bin.sort_index(axis=1))  # sort bins left-to-right

=== Overall Opportunity Rate by Attempt Bin ===
                      Events  Opps  Opp Rate %
attempt_bin                                   
16+                       77    50        64.9
Attempt 1               2494  2125        85.2
Attempts 10–12           183   130        71.0
Attempts 13–15           123    71        57.7
Attempts 1–3 (day 1)     991   766        77.3
Attempts 4–6             646   476        73.7
Attempts 7–9             302   214        70.9

=== Opportunity Rate (%) by Segment & Attempt Bin ===
attempt_bin             16+  Attempt 1  Attempts 10–12  Attempts 13–15  \
Lead Channel Segment                                                     
Digital Buy Partner    61.0       92.1            77.9            63.3   
Direct / Organic      100.0       94.4            72.7            77.8   
Movoto                 60.9       67.9            61.8            45.6   
Other                 100.0       76.2             NaN             NaN   
Paid Spend             77.8   

In [17]:
print("Number of unique leads with Transfer/Appointment:")
print(success_calls['Lead ID'].nunique())

print("\nTotal events:", len(success_calls))
print("Difference (multiple events per lead):", len(success_calls) - success_calls['Lead ID'].nunique())

Number of unique leads with Transfer/Appointment:
4526

Total events: 4816
Difference (multiple events per lead): 290


In [18]:
# Create the flag in the main leads DataFrame (if not already done)
if 'had_opportunity' not in leads.columns:
    leads['had_opportunity'] = leads['Number of Opportunities'] >= 1
    print("Created 'had_opportunity' in leads")
else:
    print("'had_opportunity' already exists in leads")

# Quick check
print(leads['had_opportunity'].value_counts(dropna=False))
# Step 1: Get FIRST success attempt per lead
first_success_per_lead = success_calls.loc[
    success_calls.groupby('Lead ID')['attempt'].idxmin()
][['Lead ID', 'attempt', 'Lead Channel Segment']]

# Step 2: Merge opportunity flag (per lead)
first_success_per_lead = first_success_per_lead.merge(
    leads[['Lead ID', 'had_opportunity']],
    on='Lead ID',
    how='left'
)

# Step 3: Apply the same bins
first_success_per_lead['attempt_bin'] = first_success_per_lead['attempt'].apply(bin_attempt)

# Step 4: Overall per bin (now unique leads)
opp_by_bin_lead = first_success_per_lead.groupby('attempt_bin')['had_opportunity'].agg(
    unique_leads='size',
    opp_count='sum',
    opp_rate=lambda x: x.mean() * 100
).round({'opp_rate': 1})

opp_by_bin_lead = opp_by_bin_lead[['unique_leads', 'opp_count', 'opp_rate']]
opp_by_bin_lead.columns = ['Unique Leads', 'Opps', 'Opp Rate %']

print("=== Opportunity Rate by Attempt Bin (strict per-lead, first success only) ===")
print(opp_by_bin_lead.sort_index())

# Optional: same by segment
opp_by_segment_bin_lead = first_success_per_lead.groupby(['Lead Channel Segment', 'attempt_bin'])['had_opportunity'].mean().unstack().mul(100).round(1)
print("\n=== Opp Rate (%) by Segment & Attempt Bin (per-lead) ===")
print(opp_by_segment_bin_lead.sort_index(axis=1))

Created 'had_opportunity' in leads
had_opportunity
False    22260
True      8756
Name: count, dtype: int64
=== Opportunity Rate by Attempt Bin (strict per-lead, first success only) ===
                      Unique Leads  Opps  Opp Rate %
attempt_bin                                         
16+                             63    44        69.8
Attempt 1                     2494  2125        85.2
Attempts 10–12                 139   100        71.9
Attempts 13–15                 102    62        60.8
Attempts 1–3 (day 1)           902   706        78.3
Attempts 4–6                   564   418        74.1
Attempts 7–9                   262   195        74.7

=== Opp Rate (%) by Segment & Attempt Bin (per-lead) ===
attempt_bin                 16+  Attempt 1 Attempts 10–12 Attempts 13–15  \
Lead Channel Segment                                                       
Digital Buy Partner   68.571429  92.116788      80.357143      63.043478   
Direct / Organic          100.0   94.42623      72.7

In [19]:
# Aggregate both count and mean in one go
segment_bin_summary = first_success_per_lead.groupby(['Lead Channel Segment', 'attempt_bin']).agg(
    unique_leads=('Lead ID', 'nunique'),
    opp_rate=('had_opportunity', 'mean')
)

# Convert opp_rate to %
segment_bin_summary['opp_rate_%'] = (segment_bin_summary['opp_rate'] * 100).round(1)

# Pivot to wide format: rows = segment, columns = bin (with unique_leads and opp_rate_% nested)
pivoted = segment_bin_summary.pivot_table(
    index='Lead Channel Segment',
    columns='attempt_bin',
    values=['unique_leads', 'opp_rate_%'],
    aggfunc='first'  # since we already aggregated
)

# Flatten multi-level columns for readability
pivoted.columns = [f"{col[1]} ({col[0]})" for col in pivoted.columns]

# Sort bins left-to-right (attempt order)
bin_order = ['Attempt 1', 'Attempts 1–3 (day 1)', 'Attempts 4–6', 'Attempts 7–9', 'Attempts 10–12', 'Attempts 13–15', '16+']
ordered_cols = [f"{b} (unique_leads)" for b in bin_order if f"{b} (unique_leads)" in pivoted.columns] + \
               [f"{b} (opp_rate_%)" for b in bin_order if f"{b} (opp_rate_%)" in pivoted.columns]

pivoted = pivoted[[c for c in ordered_cols if c in pivoted.columns]]

# Sort segments by total wins (optional: can change)
print("=== Opportunity Rate (%) and Sample Sizes (Unique Leads) by Segment & Attempt Bin ===")
print("Format: Opp Rate % (Unique Leads count)")
print(pivoted.to_string(na_rep='—'))

=== Opportunity Rate (%) and Sample Sizes (Unique Leads) by Segment & Attempt Bin ===
Format: Opp Rate % (Unique Leads count)
                      Attempt 1 (unique_leads)  Attempts 1–3 (day 1) (unique_leads)  Attempts 4–6 (unique_leads)  Attempts 7–9 (unique_leads)  Attempts 10–12 (unique_leads)  Attempts 13–15 (unique_leads)  16+ (unique_leads) Attempt 1 (opp_rate_%) Attempts 1–3 (day 1) (opp_rate_%) Attempts 4–6 (opp_rate_%) Attempts 7–9 (opp_rate_%) Attempts 10–12 (opp_rate_%) Attempts 13–15 (opp_rate_%) 16+ (opp_rate_%)
Lead Channel Segment                                                                                                                                                                                                                                                                                                                                                                                                 
Digital Buy Partner                      685.0                

In [20]:
# Ensure flags in leads
leads['had_opportunity'] = leads['Number of Opportunities'] >= 1
leads['file_started']    = leads['Number of File Started Leads'] >= 1
leads['funded']          = leads['Number of Funded Loan Leads'] >= 1

# Merge
milestone_view = first_success_per_lead.merge(
    leads[['Lead ID', 'had_opportunity', 'file_started', 'funded']],
    on='Lead ID',
    how='left'
)

# Clean up any duplicate had_opportunity columns
if 'had_opportunity_x' in milestone_view.columns:
    milestone_view['had_opportunity'] = milestone_view.get('had_opportunity_y', milestone_view['had_opportunity_x'])
    milestone_view = milestone_view.drop(columns=[c for c in milestone_view.columns if c.startswith('had_opportunity_')])

print("Columns after merge & cleanup:", milestone_view.columns.tolist())

# Filter to focus segments
focus_segments = ['Movoto', 'Digital Buy Partner']
milestone_view = milestone_view[milestone_view['Lead Channel Segment'].isin(focus_segments)]

# Aggregate
milestone_rates = milestone_view.groupby(['Lead Channel Segment', 'attempt_bin']).agg(
    unique_leads=('Lead ID', 'nunique'),
    opp_rate=('had_opportunity', 'mean'),
    file_started_rate=('file_started', 'mean'),
    funded_rate=('funded', 'mean')
).mul(100).round(1)

milestone_rates.columns = ['Unique Leads', '→ Opportunity %', '→ File Started %', '→ Funded %']

print("\n=== Movoto & Digital Buy Partner: Milestone Progression by First Success Attempt Bin ===")
print("(Rates are % of unique leads whose first success was in that bin)")
print(milestone_rates.sort_index().to_string(na_rep='—'))

Columns after merge & cleanup: ['Lead ID', 'attempt', 'Lead Channel Segment', 'attempt_bin', 'file_started', 'funded', 'had_opportunity']

=== Movoto & Digital Buy Partner: Milestone Progression by First Success Attempt Bin ===
(Rates are % of unique leads whose first success was in that bin)
                                           Unique Leads → Opportunity % → File Started % → Funded %
Lead Channel Segment attempt_bin                                                                   
Digital Buy Partner  16+                           3500       68.571429        17.142857        0.0
                     Attempt 1                    68500       92.116788        32.408759   0.145985
                     Attempts 10–12                5600       80.357143        30.357143        0.0
                     Attempts 13–15                4600       63.043478        10.869565        0.0
                     Attempts 1–3 (day 1)         21600       83.796296        28.703704        0.0
      

In [21]:
# Join calls (Pool A outbound) with leads to get segment
calls_with_segment = calls.merge(
    leads[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='inner'  # only keep calls that have segment info
)

# Filter to the two segments
focus_calls = calls_with_segment[
    calls_with_segment['Lead Channel Segment'].isin(['Movoto', 'Digital Buy Partner'])
]

# Quick sanity checks
print("Shape of focus_calls:", focus_calls.shape)
print("\nCall rows per segment:")
print(focus_calls['Lead Channel Segment'].value_counts())

print("\nUnique leads per segment:")
print(focus_calls.groupby('Lead Channel Segment')['Lead ID'].nunique())

Shape of focus_calls: (222737, 24)

Call rows per segment:
Lead Channel Segment
Digital Buy Partner    149355
Movoto                  73382
Name: count, dtype: int64

Unique leads per segment:
Lead Channel Segment
Digital Buy Partner    12201
Movoto                  8917
Name: Lead ID, dtype: int64


In [22]:
# Recreate the clean integer attempt column from the original name
focus_calls['attempt'] = focus_calls['Pool A Call Attempt Number'].astype(int, errors='ignore').fillna(0).astype(int)
# Limit to attempts 1-15 + tail for readability
focus_calls_limited = focus_calls[focus_calls['attempt'] <= 15].copy()
focus_calls_limited['attempt_bucket'] = focus_calls_limited['attempt'].astype(str)
focus_calls_limited.loc[focus_calls_limited['attempt'] > 15, 'attempt_bucket'] = '16+'

# Crosstab: rows = attempt bucket, columns = Outcome, values = count of calls
outcome_by_attempt = pd.crosstab(
    focus_calls_limited['attempt_bucket'],
    focus_calls_limited['Outcome']
)

# Add total
outcome_by_attempt['Total Calls'] = outcome_by_attempt.sum(axis=1)

# Sort attempts in logical order
attempt_order = [str(i) for i in range(1, 16)] + ['16+']
outcome_by_attempt = outcome_by_attempt.reindex(attempt_order, fill_value=0)

# Show top outcomes + total
top_outcomes = outcome_by_attempt.sum().sort_values(ascending=False).head(12).index.tolist()
if 'Total Calls' not in top_outcomes:
    top_outcomes.append('Total Calls')

print("=== Outcome Counts by Attempt Number — Movoto + Digital Buy Partner (Pool A calls) ===")
print(outcome_by_attempt[top_outcomes].to_string())

print("\nTop outcomes shown:", ', '.join(top_outcomes[:-1]))

=== Outcome Counts by Attempt Number — Movoto + Digital Buy Partner (Pool A calls) ===
Outcome         Total Calls  No Answer  Left Voicemail  Hung Up  Transfer Sent  Appointment Set  Nurture - No Benefit  Add to DNC  Nurture - Went With Another Lender  Redial  Bad/Disconnected Phone  Hold Time Exceeded
attempt_bucket                                                                                                                                                                                                           
1                     17884      10963            3511      609            950              529                   204          54                                 126     368                     101                 146
2                     13875      12172             395      408            212              187                   140          69                                  64      21                      55                  31
3                     13220      12262   

In [23]:
# leads in Movoto and Digital Buy Partner segments that eventually became opportunities (Number of Opportunities >= 1), but had at least one "No Answer" outcome in their call activity. Then, examine their call sequences to see what broke the pattern (e.g., a voicemail, a successful contact on a later attempt, or timing differences). We can compare this to similar leads that didn't convert to spot differences in persistence, attempt timing, or other factors (e.g., day of week, hour, agent role).

In [24]:
# Create opportunity flag if not already there
leads['had_opportunity'] = leads['Number of Opportunities'] >= 1

# Filter to the two segments + only leads that became opportunities
opp_leads_focus = leads[
    (leads['Lead Channel Segment'].isin(['Movoto', 'Digital Buy Partner'])) &
    (leads['had_opportunity'] == True)
].copy()

# Quick sanity checks
print("Shape of opp_leads_focus:", opp_leads_focus.shape)
print("\nCount per segment:")
print(opp_leads_focus['Lead Channel Segment'].value_counts())

print("\nUnique Lead IDs:", opp_leads_focus['Lead ID'].nunique())
print("\nSample of Current Lead Status for these leads:")
print(opp_leads_focus['Current Lead Status'].value_counts(dropna=False).head(10))

Shape of opp_leads_focus: (4928, 24)

Count per segment:
Lead Channel Segment
Digital Buy Partner    2483
Movoto                 2445
Name: count, dtype: int64

Unique Lead IDs: 4928

Sample of Current Lead Status for these leads:
Current Lead Status
Nurture                   1858
Open Transfer              922
Open Application           894
Contact Attempt            344
Application Completed      238
Calendar Event             111
Pre-Approved               108
Remarket - Uncontacted      99
Do Not Call                 81
Open Disclosures Sent       54
Name: count, dtype: int64


In [25]:
# Join opp_leads_focus with calls (keep only Pool A outbound calls for these leads)
opp_calls_history = calls[
    calls['Lead ID'].isin(opp_leads_focus['Lead ID'])
].copy()

# Add segment from opp_leads_focus for easy grouping later
opp_calls_history = opp_calls_history.merge(
    opp_leads_focus[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='left'
)

# Quick checks
print("Shape of opp_calls_history:", opp_calls_history.shape)
print("\nCall rows per segment (among opp leads):")
print(opp_calls_history['Lead Channel Segment'].value_counts(dropna=False))

print("\nUnique leads with at least one call:", opp_calls_history['Lead ID'].nunique())
print("\nAttempt range in these calls:", 
      opp_calls_history['Call Attempt Number'].min(), "to", 
      opp_calls_history['Call Attempt Number'].max())

print("\nOutcome distribution in these calls (top 10):")
print(opp_calls_history['Outcome'].value_counts(dropna=False).head(10))

Shape of opp_calls_history: (24719, 24)

Call rows per segment (among opp leads):
Lead Channel Segment
Movoto                 13241
Digital Buy Partner    11478
Name: count, dtype: int64

Unique leads with at least one call: 4675

Attempt range in these calls: 1 to 98

Outcome distribution in these calls (top 10):
Outcome
No Answer                17301
Transfer Sent             2540
Left Voicemail            1391
Hung Up                    884
Appointment Set            842
Hold Time Exceeded         242
Application Completed      225
Called - Follow Up         212
Redial                     208
Pitch Completed            147
Name: count, dtype: int64


In [26]:
# Filter to calls for opp leads that had at least one No Answer
no_answer_survivors = opp_calls_history[
    opp_calls_history['Outcome'] == 'No Answer'
].groupby('Lead ID').filter(lambda x: len(x) >= 1)

# Now get only those who also had a positive success
positive_outcomes = ['Transfer Sent', 'Appointment Set']
success_leads = opp_calls_history[
    opp_calls_history['Outcome'].isin(positive_outcomes)
]['Lead ID'].unique()

no_answer_survivors = no_answer_survivors[no_answer_survivors['Lead ID'].isin(success_leads)]

# Quick summary
print("Number of leads with at least one No Answer → eventual success + opp:")
print(no_answer_survivors['Lead ID'].nunique())

print("\nPer segment:")
print(no_answer_survivors.groupby('Lead Channel Segment')['Lead ID'].nunique())

# For these leads, compute simple stats
lead_stats = opp_calls_history[
    opp_calls_history['Lead ID'].isin(no_answer_survivors['Lead ID'].unique())
].groupby('Lead ID').agg(
    total_attempts=('Call Attempt Number', 'max'),
    no_answer_count=('Outcome', lambda x: (x == 'No Answer').sum()),
    first_success_attempt=('Call Attempt Number', lambda x: x[opp_calls_history.loc[x.index, 'Outcome'].isin(positive_outcomes)].min()),
    first_success_outcome=('Outcome', lambda x: x[opp_calls_history.loc[x.index, 'Outcome'].isin(positive_outcomes)].iloc[0] if any(opp_calls_history.loc[x.index, 'Outcome'].isin(positive_outcomes)) else None)
)

print("\nSummary stats for No Answer survivors (leads that converted despite No Answer):")
print(lead_stats.describe().round(1))

print("\nFirst success outcome distribution:")
print(lead_stats['first_success_outcome'].value_counts(normalize=True).mul(100).round(1))

Number of leads with at least one No Answer → eventual success + opp:
1808

Per segment:
Lead Channel Segment
Digital Buy Partner    893
Movoto                 915
Name: Lead ID, dtype: int64

Summary stats for No Answer survivors (leads that converted despite No Answer):
       total_attempts  no_answer_count  first_success_attempt
count          1808.0           1808.0                 1808.0
mean              8.6              4.9                    4.6
std               6.2              4.8                    4.2
min               2.0              1.0                    1.0
25%               4.0              1.0                    2.0
50%               7.0              3.0                    3.0
75%              11.0              7.0                    6.0
max              98.0             33.0                   33.0

First success outcome distribution:
first_success_outcome
Transfer Sent      74.4
Appointment Set    25.6
Name: proportion, dtype: float64


In [28]:
# Distribution of first success attempt (histogram-like value counts)
print("Distribution of attempt number when first success happened (Transfer or Appt Set):")
print(lead_stats['first_success_attempt'].value_counts().sort_index().head(20))

print("\nCumulative % of survivors who succeeded by attempt N:")
cumulative = lead_stats['first_success_attempt'].value_counts().sort_index().cumsum() / len(lead_stats) * 100
print(cumulative.round(1).head(15))

print("\nMedian / Mean / 90th percentile attempts to first success:")
# print(lead_stats['first_success_attempt'].describe()[['50%', 'mean', '90%']].round(1))

Distribution of attempt number when first success happened (Transfer or Appt Set):
first_success_attempt
1     445
2     309
3     230
4     166
5     119
6     113
7      86
8      65
9      45
10     42
11     44
12     23
13     27
14     25
15     15
16      9
17     10
18     10
19      7
20      9
Name: count, dtype: int64

Cumulative % of survivors who succeeded by attempt N:
first_success_attempt
1     24.6
2     41.7
3     54.4
4     63.6
5     70.2
6     76.4
7     81.2
8     84.8
9     87.3
10    89.6
11    92.0
12    93.3
13    94.8
14    96.2
15    97.0
Name: count, dtype: float64

Median / Mean / 90th percentile attempts to first success:


In [37]:
# Step 1: Rebuild survivor_leads from lead_stats (the 1,808 IDs)
survivor_leads = lead_stats.index.tolist()  # from earlier lead_stats

# Step 2: Rebuild survivor_calls = all calls for these leads
survivor_calls = calls[calls['Lead ID'].isin(survivor_leads)].copy()

# Step 3: Sort by lead + time (required for sequence analysis)
survivor_calls['Activity (ET) Time'] = pd.to_datetime(survivor_calls['Activity (ET) Time'], errors='coerce')
survivor_calls_sorted = survivor_calls.sort_values(['Lead ID', 'Activity (ET) Time'])

# Step 4: Find first success per lead (Transfer or Appt Set)
first_success = survivor_calls_sorted[
    survivor_calls_sorted['Outcome'].isin(['Transfer Sent', 'Appointment Set'])
].groupby('Lead ID')['Activity (ET) Time'].min().reset_index()
first_success = first_success.rename(columns={'Activity (ET) Time': 'first_success_time'})

# Step 5: Get No Answer calls BEFORE first success
no_answer_before_success = survivor_calls_sorted.merge(
    first_success,
    on='Lead ID',
    how='inner'
)
no_answer_before_success = no_answer_before_success[
    (no_answer_before_success['Outcome'] == 'No Answer') &
    (no_answer_before_success['Activity (ET) Time'] < no_answer_before_success['first_success_time'])
]

# Step 6: Last No Answer before success
last_no_answer_correct = no_answer_before_success.groupby('Lead ID')['Activity (ET) Time'].max().reset_index()
last_no_answer_correct = last_no_answer_correct.rename(columns={'Activity (ET) Time': 'last_no_answer_before_success'})

# Step 7: Merge and calculate deltas
time_gaps_correct = last_no_answer_correct.merge(
    first_success,
    on='Lead ID',
    how='inner'
)
time_gaps_correct = time_gaps_correct.merge(
    leads[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='left'
)

time_gaps_correct['hours_to_success'] = (time_gaps_correct['first_success_time'] - time_gaps_correct['last_no_answer_before_success']).dt.total_seconds() / 3600
time_gaps_correct['days_to_success'] = time_gaps_correct['hours_to_success'] / 24

# Step 8: Output
print("Number of leads with valid pre-success last No Answer:", len(time_gaps_correct))
print("\nCorrected time gaps summary (hours):")
print(time_gaps_correct['hours_to_success'].describe().round(1))

print("\nCorrected time gaps summary (days):")
print(time_gaps_correct['days_to_success'].describe().round(1))

print("\nPer segment corrected time gaps (hours, median / mean):")
print(time_gaps_correct.groupby('Lead Channel Segment')['hours_to_success'].agg(['count', 'median', 'mean']).round(1))

Number of leads with valid pre-success last No Answer: 1258

Corrected time gaps summary (hours):
count    1258.0
mean       23.5
std        56.9
min         0.0
25%         3.2
50%         4.0
75%        20.3
max       767.2
Name: hours_to_success, dtype: float64

Corrected time gaps summary (days):
count    1258.0
mean        1.0
std         2.4
min         0.0
25%         0.1
50%         0.2
75%         0.8
max        32.0
Name: days_to_success, dtype: float64

Per segment corrected time gaps (hours, median / mean):
                      count  median  mean
Lead Channel Segment                     
Digital Buy Partner     626     4.0  21.5
Movoto                  632     4.1  25.5


In [38]:
# Ensure Activity Time is datetime
survivor_calls['Activity (ET) Time'] = pd.to_datetime(survivor_calls['Activity (ET) Time'], errors='coerce')

# Add day of week (0 = Monday, 6 = Sunday) and hour (0-23)
survivor_calls['day_of_week'] = survivor_calls['Activity (ET) Time'].dt.dayofweek
survivor_calls['hour_of_day'] = survivor_calls['Activity (ET) Time'].dt.hour

# Filter to breakthrough attempts (first Transfer or Appt Set per lead)
breakthrough_calls = survivor_calls[
    survivor_calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])
].sort_values(['Lead ID', 'Activity (ET) Time'])
breakthrough_calls = breakthrough_calls.groupby('Lead ID').head(1)  # first success only

# Filter to No Answer attempts for comparison
no_answer_calls = survivor_calls[survivor_calls['Outcome'] == 'No Answer']

# Quick distribution - breakthrough day/hour
print("Breakthrough attempts - Day of Week distribution (0=Mon, 6=Sun):")
print(breakthrough_calls['day_of_week'].value_counts(normalize=True).mul(100).round(1).sort_index())

print("\nBreakthrough attempts - Hour of Day distribution (0-23):")
print(breakthrough_calls['hour_of_day'].value_counts(normalize=True).mul(100).round(1).sort_index().head(15))

# Compare to No Answer attempts
print("\nNo Answer attempts - Day of Week distribution:")
print(no_answer_calls['day_of_week'].value_counts(normalize=True).mul(100).round(1).sort_index())

print("\nNo Answer attempts - Hour of Day distribution:")
print(no_answer_calls['hour_of_day'].value_counts(normalize=True).mul(100).round(1).sort_index().head(15))

Breakthrough attempts - Day of Week distribution (0=Mon, 6=Sun):
day_of_week
0    21.4
1    21.4
2    21.2
3    17.6
4    12.9
5     2.8
6     2.6
Name: proportion, dtype: float64

Breakthrough attempts - Hour of Day distribution (0-23):
hour_of_day
8      4.9
9     15.7
10    11.8
11     9.5
12    10.2
13    10.9
14    10.5
15     9.8
16     9.1
17     4.7
18     2.5
19     0.5
Name: proportion, dtype: float64

No Answer attempts - Day of Week distribution:
day_of_week
0    19.0
1    22.5
2    20.1
3    17.1
4    14.8
5     3.3
6     3.2
Name: proportion, dtype: float64

No Answer attempts - Hour of Day distribution:
hour_of_day
8      4.8
9     13.4
10    14.3
11    10.6
12     9.0
13     8.8
14    10.5
15    10.5
16     9.8
17     5.8
18     2.2
19     0.2
20     0.0
Name: proportion, dtype: float64


In [39]:
# Breakthrough calls (first success per lead)
breakthrough_calls = survivor_calls[
    survivor_calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])
].sort_values(['Lead ID', 'Activity (ET) Time'])
breakthrough_calls = breakthrough_calls.groupby('Lead ID').head(1)  # only first success

# No Answer calls (prior to success)
no_answer_prior = survivor_calls[
    (survivor_calls['Outcome'] == 'No Answer') &
    (survivor_calls['Lead ID'].isin(breakthrough_calls['Lead ID']))
]

# Quick distribution of roles on breakthrough calls
print("Agent Role distribution on breakthrough calls:")
print(breakthrough_calls['Team Member Role'].value_counts(normalize=True).mul(100).round(1))

print("\nTop 10 Team Member Roles (count) on breakthrough calls:")
print(breakthrough_calls['Team Member Role'].value_counts().head(10))

# Compare to No Answer calls
print("\nAgent Role distribution on prior No Answer calls:")
print(no_answer_prior['Team Member Role'].value_counts(normalize=True).mul(100).round(1))

print("\nTop 10 Team Member Roles (count) on No Answer calls:")
print(no_answer_prior['Team Member Role'].value_counts().head(10))

Agent Role distribution on breakthrough calls:
Team Member Role
Customer Engagement Representative Level Two    92.7
Loan Advisor                                     5.0
Sr Sales Development Representative              1.0
Sales Development Representative                 0.9
Loan Prospector                                  0.2
AVP of Sales                                     0.1
Contact Center Manager                           0.1
Name: proportion, dtype: float64

Top 10 Team Member Roles (count) on breakthrough calls:
Team Member Role
Customer Engagement Representative Level Two    1676
Loan Advisor                                      90
Sr Sales Development Representative               18
Sales Development Representative                  17
Loan Prospector                                    4
AVP of Sales                                       2
Contact Center Manager                             1
Name: count, dtype: int64

Agent Role distribution on prior No Answer calls:
Team Membe

In [40]:
# Count breakthrough successes per agent
agent_breakthroughs = breakthrough_calls.groupby(['Team Member ID', 'Team Member Name', 'Team Member Role']).size().reset_index(name='breakthrough_count')

# Sort descending by count
agent_breakthroughs = agent_breakthroughs.sort_values('breakthrough_count', ascending=False)

# Add total breakthroughs for context
total_breakthroughs = agent_breakthroughs['breakthrough_count'].sum()

print("Total breakthrough calls in survivors:", total_breakthroughs)
print("\nTop 10 agents by breakthrough count:")
print(agent_breakthroughs.head(10)[['Team Member Name', 'Team Member Role', 'breakthrough_count']])

print("\nAgents with 20+ breakthroughs (top performers):")
print(agent_breakthroughs[agent_breakthroughs['breakthrough_count'] >= 20][['Team Member Name', 'breakthrough_count']])

print("\nNumber of unique agents who had at least one breakthrough:", agent_breakthroughs['Team Member ID'].nunique())
print("Median breakthroughs per agent:", agent_breakthroughs['breakthrough_count'].median())

Total breakthrough calls in survivors: 1808

Top 10 agents by breakthrough count:
      Team Member Name                              Team Member Role  \
17       Sunun Matthew  Customer Engagement Representative Level Two   
32      Kelda McDonald  Customer Engagement Representative Level Two   
37       Kenwin Joseph  Customer Engagement Representative Level Two   
20    Shannon Marshall  Customer Engagement Representative Level Two   
30         Frani Henry  Customer Engagement Representative Level Two   
29    Quetanah Modeste  Customer Engagement Representative Level Two   
31       Zelda Laurent  Customer Engagement Representative Level Two   
33      Merleana Burke  Customer Engagement Representative Level Two   
34          Zenia Paul  Customer Engagement Representative Level Two   
40  Nathline Duplessis  Customer Engagement Representative Level Two   

    breakthrough_count  
17                 142  
32                 140  
37                 138  
20                 132  


In [41]:
# Total calls per agent (among survivor leads' calls)
agent_total_calls = survivor_calls.groupby(['Team Member ID', 'Team Member Name', 'Team Member Role']).size().reset_index(name='total_calls')

# Breakthroughs per agent (from earlier)
agent_breakthroughs = breakthrough_calls.groupby(['Team Member ID', 'Team Member Name', 'Team Member Role']).size().reset_index(name='breakthrough_count')

# Merge to get both metrics
agent_performance = agent_total_calls.merge(
    agent_breakthroughs,
    on=['Team Member ID', 'Team Member Name', 'Team Member Role'],
    how='left'
).fillna({'breakthrough_count': 0})

# Calculate success rate
agent_performance['success_rate'] = (agent_performance['breakthrough_count'] / agent_performance['total_calls'] * 100).round(2)

# Sort by success rate descending (only agents with >=10 total calls to avoid noise)
agent_performance_filtered = agent_performance[agent_performance['total_calls'] >= 10].sort_values('success_rate', ascending=False)

print("Agents sorted by success rate (breakthroughs / total calls) — min 10 calls:")
print(agent_performance_filtered[['Team Member Name', 'Team Member Role', 'total_calls', 'breakthrough_count', 'success_rate']].head(15))

print("\nTop 5 agents (by count) success rate:")
top5_names = agent_breakthroughs.sort_values('breakthrough_count', ascending=False).head(5)['Team Member Name'].tolist()
print(agent_performance[agent_performance['Team Member Name'].isin(top5_names)][['Team Member Name', 'total_calls', 'breakthrough_count', 'success_rate']])

print("\nOverall average success rate (all agents with >=10 calls):", 
      agent_performance_filtered['success_rate'].mean().round(2), "%")
print("Median success rate:", agent_performance_filtered['success_rate'].median().round(2), "%")

Agents sorted by success rate (breakthroughs / total calls) — min 10 calls:
       Team Member Name                              Team Member Role  \
12        Holly Seberig           Sr Sales Development Representative   
77     Quetanah Modeste  Customer Engagement Representative Level Two   
68         Connor Bourn                                  Loan Advisor   
80       Kelda McDonald  Customer Engagement Representative Level Two   
78          Frani Henry  Customer Engagement Representative Level Two   
73    Nikita Lartchenko                                  Loan Advisor   
35         Brae Humbert              Sales Development Representative   
97       Lisa St. Ville  Customer Engagement Representative Level Two   
60        Sunun Matthew  Customer Engagement Representative Level Two   
96       Bernetta Felix  Customer Engagement Representative Level Two   
65        Lucas Dickson                                  Loan Advisor   
81       Merleana Burke  Customer Engagement Rep

In [42]:
# Filter agent_performance to only CERR Level Two
cerr_l2_performance = agent_performance[
    agent_performance['Team Member Role'] == 'Customer Engagement Representative Level Two'
].copy()

# Re-sort by success rate descending (still min 10 calls)
cerr_l2_performance = cerr_l2_performance[cerr_l2_performance['total_calls'] >= 10].sort_values('success_rate', ascending=False)

print("Number of CERR L2 agents with >=10 calls:", len(cerr_l2_performance))
print("\nTop 5 CERR L2 by success rate:")
print(cerr_l2_performance.head(5)[['Team Member Name', 'total_calls', 'breakthrough_count', 'success_rate']])

print("\nBottom 20 CERR L2 by success rate (last 20 rows):")
print(cerr_l2_performance.tail(20)[['Team Member Name', 'total_calls', 'breakthrough_count', 'success_rate']])

print("\nOverall for CERR L2:")
print("Average success rate:", cerr_l2_performance['success_rate'].mean().round(2), "%")
print("Median success rate:", cerr_l2_performance['success_rate'].median().round(2), "%")

Number of CERR L2 agents with >=10 calls: 17

Top 5 CERR L2 by success rate:
    Team Member Name  total_calls  breakthrough_count  success_rate
77  Quetanah Modeste          536               123.0         22.95
80    Kelda McDonald          652               140.0         21.47
78       Frani Henry          605               129.0         21.32
97    Lisa St. Ville          120                24.0         20.00
60     Sunun Matthew          727               142.0         19.53

Bottom 20 CERR L2 by success rate (last 20 rows):
       Team Member Name  total_calls  breakthrough_count  success_rate
77     Quetanah Modeste          536               123.0         22.95
80       Kelda McDonald          652               140.0         21.47
78          Frani Henry          605               129.0         21.32
97       Lisa St. Ville          120                24.0         20.00
60        Sunun Matthew          727               142.0         19.53
96       Bernetta Felix           72  

In [ ]:
#######################################################

In [43]:
# Ensure flags in leads
leads['was_contacted']     = leads['Number of Contacted Leads'] >= 1
leads['had_opportunity']   = leads['Number of Opportunities'] >= 1

# Filter to the three roles we're focusing on (for the success event)
focus_roles = [
    'Customer Engagement Representative Level Two',
    'Sales Development Representative',
    'Sr Sales Development Representative'
]

# Quick check of contacted + opp in all segments (sanity)
print("Total leads with Number of Contacted Leads >=1:", leads['was_contacted'].sum())
print("Of which have opportunity:", leads[leads['was_contacted']]['had_opportunity'].sum())
print("Overall opp rate among contacted leads:", 
      (leads[leads['was_contacted']]['had_opportunity'].mean() * 100).round(1), "%")

Total leads with Number of Contacted Leads >=1: 17589
Of which have opportunity: 7708
Overall opp rate among contacted leads: 43.8 %


In [44]:
# Define the 3 focus roles exactly as they appear
focus_roles = [
    'Customer Engagement Representative Level Two',
    'Sales Development Representative',
    'Sr Sales Development Representative'
]

# Filter breakthrough_calls (first success per lead) to only those 3 roles
breakthrough_focus_roles = breakthrough_calls[
    breakthrough_calls['Team Member Role'].isin(focus_roles)
].copy()

# Quick check
print("Number of first-success events by focus roles:", len(breakthrough_focus_roles))
print("\nBreakdown by role:")
print(breakthrough_focus_roles['Team Member Role'].value_counts())

print("\nUnique leads with success by these roles:", breakthrough_focus_roles['Lead ID'].nunique())

Number of first-success events by focus roles: 1711

Breakdown by role:
Team Member Role
Customer Engagement Representative Level Two    1676
Sr Sales Development Representative               18
Sales Development Representative                  17
Name: count, dtype: int64

Unique leads with success by these roles: 1711


In [45]:
# Add the attempt column (fix for KeyError)
breakthrough_focus_roles['attempt'] = breakthrough_focus_roles['Pool A Call Attempt Number'].astype(int, errors='ignore').fillna(0).astype(int)

# Add attempt bin
breakthrough_focus_roles['attempt_bin'] = breakthrough_focus_roles['attempt'].apply(bin_attempt)

# Join to leads for had_opportunity
breakthrough_with_opp = breakthrough_focus_roles.merge(
    leads[['Lead ID', 'had_opportunity']],
    on='Lead ID',
    how='left'
)

# Aggregate opp rate by bin
opp_rate_by_bin_role = breakthrough_with_opp.groupby('attempt_bin').agg(
    unique_leads=('Lead ID', 'nunique'),
    opp_count=('had_opportunity', 'sum'),
    opp_rate=('had_opportunity', 'mean')
).reset_index()

opp_rate_by_bin_role['opp_rate_%'] = (opp_rate_by_bin_role['opp_rate'] * 100).round(1)

# Sort bins in logical order
bin_order = ['Attempt 1', 'Attempts 1–3 (day 1)', 'Attempts 4–6', 'Attempts 7–9', 'Attempts 10–12', 'Attempts 13–15', '16+']
opp_rate_by_bin_role['attempt_bin'] = pd.Categorical(opp_rate_by_bin_role['attempt_bin'], categories=bin_order, ordered=True)
opp_rate_by_bin_role = opp_rate_by_bin_role.sort_values('attempt_bin')

print("=== Opportunity Rate by Attempt Bin — First Success by CERR L2 / SDR / Sr SDR ===")
print("(Among leads where first contact was by one of these 3 roles)")
print(opp_rate_by_bin_role[['attempt_bin', 'unique_leads', 'opp_count', 'opp_rate_%']].to_string(index=False))

=== Opportunity Rate by Attempt Bin — First Success by CERR L2 / SDR / Sr SDR ===
(Among leads where first contact was by one of these 3 roles)
         attempt_bin  unique_leads  opp_count  opp_rate_%
           Attempt 1           332        332       100.0
Attempts 1–3 (day 1)           363        363       100.0
        Attempts 4–6           267        267       100.0
        Attempts 7–9           143        143       100.0
      Attempts 10–12            69         69       100.0
      Attempts 13–15            44         44       100.0
                 16+           493        493       100.0


In [46]:
# Ensure flags in leads (idempotent)
leads['was_contacted']   = leads['Number of Contacted Leads'] >= 1
leads['had_opportunity'] = leads['Number of Opportunities'] >= 1

# Define the exact role names
focus_roles = [
    'Customer Engagement Representative Level Two',
    'Sales Development Representative',
    'Sr Sales Development Representative'
]

# Filter breakthrough_calls to only these roles
breakthrough_focus_roles = breakthrough_calls[
    breakthrough_calls['Team Member Role'].isin(focus_roles)
].copy()

# Add attempt column (in case it was missing)
breakthrough_focus_roles['attempt'] = breakthrough_focus_roles['Pool A Call Attempt Number'].astype(int, errors='ignore').fillna(0).astype(int)

print("Number of first-success events by these 3 roles:", len(breakthrough_focus_roles))
print("\nBreakdown by role:")
print(breakthrough_focus_roles['Team Member Role'].value_counts(dropna=False))

print("\nUnique leads with first success by these roles:", breakthrough_focus_roles['Lead ID'].nunique())

Number of first-success events by these 3 roles: 1711

Breakdown by role:
Team Member Role
Customer Engagement Representative Level Two    1676
Sr Sales Development Representative               18
Sales Development Representative                  17
Name: count, dtype: int64

Unique leads with first success by these roles: 1711


In [47]:
# Add attempt column if missing (from Pool A Call Attempt Number)
breakthrough_focus_roles['attempt'] = breakthrough_focus_roles['Pool A Call Attempt Number'].astype(int, errors='ignore').fillna(0).astype(int)

# Apply the same attempt bin function (assuming bin_attempt is defined from earlier)
breakthrough_focus_roles['attempt_bin'] = breakthrough_focus_roles['attempt'].apply(bin_attempt)

# Merge with leads to bring in had_opportunity (lead-level flag)
breakthrough_with_opp = breakthrough_focus_roles.merge(
    leads[['Lead ID', 'had_opportunity']],
    on='Lead ID',
    how='left'
)

# Aggregate opp rate by attempt bin
opp_rate_by_bin = breakthrough_with_opp.groupby('attempt_bin').agg(
    unique_leads=('Lead ID', 'nunique'),
    opp_count=('had_opportunity', 'sum'),
    opp_rate=('had_opportunity', 'mean')
).reset_index()

opp_rate_by_bin['opp_rate_%'] = (opp_rate_by_bin['opp_rate'] * 100).round(1)

# Sort bins in logical order
bin_order = ['Attempt 1', 'Attempts 1–3 (day 1)', 'Attempts 4–6', 'Attempts 7–9', 'Attempts 10–12', 'Attempts 13–15', '16+']
opp_rate_by_bin['attempt_bin'] = pd.Categorical(opp_rate_by_bin['attempt_bin'], categories=bin_order, ordered=True)
opp_rate_by_bin = opp_rate_by_bin.sort_values('attempt_bin')

print("=== Opportunity Rate by Attempt Bin — First Success by CERR L2 / SDR / Sr SDR ===")
print("(Among leads where first contact was by one of these 3 roles)")
print(opp_rate_by_bin[['attempt_bin', 'unique_leads', 'opp_count', 'opp_rate_%']].to_string(index=False))

=== Opportunity Rate by Attempt Bin — First Success by CERR L2 / SDR / Sr SDR ===
(Among leads where first contact was by one of these 3 roles)
         attempt_bin  unique_leads  opp_count  opp_rate_%
           Attempt 1           332        332       100.0
Attempts 1–3 (day 1)           363        363       100.0
        Attempts 4–6           267        267       100.0
        Attempts 7–9           143        143       100.0
      Attempts 10–12            69         69       100.0
      Attempts 13–15            44         44       100.0
                 16+           493        493       100.0


In [48]:
# Ensure the timestamp columns are datetime
focus_calls['Activity (ET) Time'] = pd.to_datetime(focus_calls['Activity (ET) Time'], errors='coerce')
focus_calls['Lead Created (ET) Time'] = pd.to_datetime(focus_calls['Lead Created (ET) Time'], errors='coerce')

# Create date columns (this is safe)
focus_calls['activity_date'] = focus_calls['Activity (ET) Time'].dt.date
focus_calls['lead_created_date'] = focus_calls['Lead Created (ET) Time'].dt.date

# Subtract dates to get timedelta, then extract days (no .dt needed here)
focus_calls['day_relative'] = (focus_calls['activity_date'] - focus_calls['lead_created_date']).apply(lambda x: x.days if pd.notnull(x) else pd.NA) + 1

# Clean up invalid rows (negative days or NaN)
focus_calls = focus_calls[focus_calls['day_relative'] >= 1].dropna(subset=['day_relative'])

# Quick checks
print("Shape of focus_calls after cleanup:", focus_calls.shape)
print("\nDay relative distribution (first 10 days):")
print(focus_calls['day_relative'].value_counts().sort_index().head(10))

print("\nCalls per day for first 5 days:")
print(focus_calls[focus_calls['day_relative'] <= 5]['day_relative'].value_counts().sort_index())

print("\nMax day_relative observed:", focus_calls['day_relative'].max())
print("Any calls on day 1:", (focus_calls['day_relative'] == 1).sum())

Shape of focus_calls after cleanup: (222715, 28)

Day relative distribution (first 10 days):
day_relative
1     29420
2     35692
3     21797
4     15716
5     14894
6     14158
7     13134
8     10141
9      6029
10     3710
Name: count, dtype: int64

Calls per day for first 5 days:
day_relative
1    29420
2    35692
3    21797
4    15716
5    14894
Name: count, dtype: int64

Max day_relative observed: 103
Any calls on day 1: 29420


In [49]:
# Ensure flag
leads['had_opportunity'] = leads['Number of Opportunities'] >= 1

# Group by channel segment
opp_by_segment = leads.groupby('Lead Channel Segment').agg(
    total_leads=('Lead ID', 'nunique'),
    contacted_leads=('was_contacted', 'sum'),
    opportunity_leads=('had_opportunity', 'sum')
)

opp_by_segment['opp_rate_%'] = (opp_by_segment['opportunity_leads'] / opp_by_segment['total_leads'] * 100).round(1)

# Sort by opportunity rate descending
opp_by_segment = opp_by_segment.sort_values('opp_rate_%', ascending=False)

print("=== Opportunity Rate by Lead Channel Segment ===")
print("Columns: Total Leads | Contacted Leads | Opportunity Leads | Opp Rate %")
print(opp_by_segment.to_string())

=== Opportunity Rate by Lead Channel Segment ===
Columns: Total Leads | Contacted Leads | Opportunity Leads | Opp Rate %
                      total_leads  contacted_leads  opportunity_leads  opp_rate_%
Lead Channel Segment                                                             
Paid Spend                   5138             2934               2275        44.3
Direct / Organic             3300             1850               1438        43.6
Other                         124               74                 54        43.5
Movoto                       9236             6638               2445        26.5
Digital Buy Partner         12509             5386               2483        19.8
Retargeting                   509              509                 47         9.2
Mail                          200              198                 14         7.0


In [50]:
# Ensure flags (idempotent)
leads['was_contacted'] = leads['Number of Contacted Leads'] >= 1
leads['had_opportunity'] = leads['Number of Opportunities'] >= 1

# Filter: two segments + contacted + NOT opportunity
no_opp_contacted = leads[
    (leads['Lead Channel Segment'].isin(['Movoto', 'Digital Buy Partner'])) &
    (leads['was_contacted'] == True) &
    (leads['had_opportunity'] == False)
].copy()

# Quick sanity check
print("Shape of no_opp_contacted:", no_opp_contacted.shape)
print("\nCount per segment:")
print(no_opp_contacted['Lead Channel Segment'].value_counts())

print("\nUnique Lead IDs:", no_opp_contacted['Lead ID'].nunique())
print("\nSample of Current Lead Status (top 5):")
print(no_opp_contacted['Current Lead Status'].value_counts(dropna=False).head())

Shape of no_opp_contacted: (7653, 24)

Count per segment:
Lead Channel Segment
Movoto                 4394
Digital Buy Partner    3259
Name: count, dtype: int64

Unique Lead IDs: 7653

Sample of Current Lead Status (top 5):
Current Lead Status
Nurture                   3737
Contact Attempt           2039
Do Not Call                845
Remarket - Uncontacted     718
Bad Phone                   96
Name: count, dtype: int64


In [51]:
# Join no_opp_contacted leads with their calls (only Pool A outbound)
no_opp_calls = calls[
    calls['Lead ID'].isin(no_opp_contacted['Lead ID'])
].copy()

# Add segment from no_opp_contacted for easy grouping
no_opp_calls = no_opp_calls.merge(
    no_opp_contacted[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='left'
)

# Quick sanity checks
print("Shape of no_opp_calls:", no_opp_calls.shape)
print("\nCall rows per segment:")
print(no_opp_calls['Lead Channel Segment'].value_counts(dropna=False))

print("\nUnique leads with at least one call:", no_opp_calls['Lead ID'].nunique())

print("\nOutcome distribution in these calls (top 10):")
print(no_opp_calls['Outcome'].value_counts(dropna=False).head(10))

Shape of no_opp_calls: (67245, 24)

Call rows per segment:
Lead Channel Segment
Digital Buy Partner    35152
Movoto                 32093
Name: count, dtype: int64

Unique leads with at least one call: 7500

Outcome distribution in these calls (top 10):
Outcome
No Answer                             56119
Hung Up                                3425
Left Voicemail                         1930
Nurture - No Benefit                   1425
Appointment Set                        1124
Redial                                  653
Nurture - Went With Another Lender      607
Add to DNC                              607
Nurture - Credit Score                  232
Nurture - Income                        190
Name: count, dtype: int64


In [52]:
# Leads that are contacted but have zero Transfer Sent or Appointment Set in calls
contacted_no_success = leads[
    (leads['was_contacted'] == True) &
    (~leads['Lead ID'].isin(
        calls[calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])]['Lead ID'].unique()
    ))
]

print("Number of contacted leads with NO Transfer Sent or Appt Set in calls:", len(contacted_no_success))
print("Their opp rate:", (contacted_no_success['had_opportunity'].mean() * 100).round(1), "%")

Number of contacted leads with NO Transfer Sent or Appt Set in calls: 11472
Their opp rate: 24.7 %


In [53]:
# Overall No Answer rate across all calls
overall_no_answer_rate = (calls['Outcome'] == 'No Answer').mean() * 100
print("Overall No Answer rate (all Pool A outbound calls, all segments):", round(overall_no_answer_rate, 1), "%")

# Number of calls total
print("Total outbound Pool A calls:", len(calls))

Overall No Answer rate (all Pool A outbound calls, all segments): 89.0 %
Total outbound Pool A calls: 289108


In [54]:
# Join calls (Pool A outbound) with leads to get segment
calls_with_segment = calls.merge(
    leads[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='inner'  # keep only matched calls
)

# Quick volume check
print("Shape of calls_with_segment:", calls_with_segment.shape)
print("\nCalls per segment:")
print(calls_with_segment['Lead Channel Segment'].value_counts(dropna=False))

# Compute No Answer rate per segment
no_answer_by_segment = calls_with_segment.groupby('Lead Channel Segment').apply(
    lambda x: ((x['Outcome'] == 'No Answer').sum() / len(x) * 100).round(1)
).reset_index(name='no_answer_%')

no_answer_by_segment['total_calls'] = calls_with_segment.groupby('Lead Channel Segment').size().values

print("\n=== No Answer % by Lead Channel Segment ===")
print(no_answer_by_segment.sort_values('no_answer_%', ascending=False).to_string(index=False))

Shape of calls_with_segment: (289078, 24)

Calls per segment:
Lead Channel Segment
Digital Buy Partner    149355
Movoto                  73382
Paid Spend              44095
Direct / Organic        19906
Other                    1144
Retargeting              1101
Mail                       95
Name: count, dtype: int64

=== No Answer % by Lead Channel Segment ===
Lead Channel Segment  no_answer_%  total_calls
 Digital Buy Partner         92.5       149355
          Paid Spend         87.7        44095
               Other         86.7         1144
              Movoto         84.8        73382
    Direct / Organic         84.2        19906
         Retargeting         35.8         1101
                Mail         16.8           95


In [55]:
# Ensure flags (idempotent)
leads['had_opportunity']   = leads['Number of Opportunities'] >= 1
leads['file_started']      = leads['Number of File Started Leads'] >= 1
leads['funded']            = leads['Number of Funded Loan Leads'] >= 1

# Group by segment and count each stage
funnel_by_segment = leads.groupby('Lead Channel Segment').agg(
    total_leads        = ('Lead ID', 'nunique'),
    contacted_leads    = ('was_contacted', 'sum'),
    opportunities      = ('had_opportunity', 'sum'),
    file_started       = ('file_started', 'sum'),
    funded             = ('funded', 'sum')
)

# Add rates (as % of previous stage)
funnel_by_segment['contact_rate_%']     = (funnel_by_segment['contacted_leads'] / funnel_by_segment['total_leads'] * 100).round(1)
funnel_by_segment['opp_rate_%']         = (funnel_by_segment['opportunities'] / funnel_by_segment['contacted_leads'] * 100).round(1)
funnel_by_segment['file_rate_%']        = (funnel_by_segment['file_started'] / funnel_by_segment['opportunities'] * 100).round(1)
funnel_by_segment['funded_rate_%']      = (funnel_by_segment['funded'] / funnel_by_segment['file_started'] * 100).round(1)

# Reorder columns for funnel look
funnel_by_segment = funnel_by_segment[[
    'total_leads', 'contacted_leads', 'opportunities', 'file_started', 'funded',
    'contact_rate_%', 'opp_rate_%', 'file_rate_%', 'funded_rate_%'
]]

print("=== Funnel by Lead Channel Segment ===")
print("Leads → Contacted → Opportunity → File Started → Funded")
print(funnel_by_segment.to_string())

=== Funnel by Lead Channel Segment ===
Leads → Contacted → Opportunity → File Started → Funded
                      total_leads  contacted_leads  opportunities  file_started  funded  contact_rate_%  opp_rate_%  file_rate_%  funded_rate_%
Lead Channel Segment                                                                                                                           
Digital Buy Partner         12509             5386           2483           762      15            43.1        46.1         30.7            2.0
Direct / Organic             3300             1850           1438           914      84            56.1        77.7         63.6            9.2
Mail                          200              198             14            57       1            99.0         7.1        407.1            1.8
Movoto                       9236             6638           2445           605       7            71.9        36.8         24.7            1.2
Other                         124        

In [56]:
# Join calls with segment from leads (inner join to keep only matched)
calls_with_segment = calls.merge(
    leads[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='inner'
)

# Quick check
print("Shape after join:", calls_with_segment.shape)
print("\nCalls per segment (top 5):")
print(calls_with_segment['Lead Channel Segment'].value_counts().head())

Shape after join: (289078, 24)

Calls per segment (top 5):
Lead Channel Segment
Digital Buy Partner    149355
Movoto                  73382
Paid Spend              44095
Direct / Organic        19906
Other                    1144
Name: count, dtype: int64


In [57]:
# Filter to only Movoto and Digital Buy Partner
focus_segments = ['Movoto', 'Digital Buy Partner']
focus_calls = calls_with_segment[calls_with_segment['Lead Channel Segment'].isin(focus_segments)].copy()

# Ensure attempt is integer (from Pool A Call Attempt Number)
focus_calls['attempt'] = focus_calls['Pool A Call Attempt Number'].astype(int, errors='ignore').fillna(0).astype(int)

# Group by attempt and compute No Answer %
no_answer_per_attempt = focus_calls.groupby('attempt').agg(
    total_calls=('Lead ID', 'count'),
    no_answer_count=('Outcome', lambda x: (x == 'No Answer').sum())
)

no_answer_per_attempt['no_answer_%'] = (no_answer_per_attempt['no_answer_count'] / no_answer_per_attempt['total_calls'] * 100).round(1)

# Limit to attempts 1–20 (tail is thin after that)
no_answer_per_attempt = no_answer_per_attempt[no_answer_per_attempt.index <= 20]

print("=== No Answer % per Attempt Number — Movoto + Digital Buy Partner ===")
print("Attempt | Total Calls | No Answer Count | No Answer %")
print(no_answer_per_attempt.to_string())

=== No Answer % per Attempt Number — Movoto + Digital Buy Partner ===
Attempt | Total Calls | No Answer Count | No Answer %
         total_calls  no_answer_count  no_answer_%
attempt                                           
0              31881            23998         75.3
1              17884            10963         61.3
2              13875            12172         87.7
3              13220            12262         92.8
4              13583            12780         94.1
5              13127            12462         94.9
6              12404            11885         95.8
7              11758            11343         96.5
8              11221            10862         96.8
9              10796            10477         97.0
10             10297            10005         97.2
11              9924             9641         97.1
12              9498             9242         97.3
13              9113             8893         97.6
14              8670             8484         97.9
15       

In [58]:
# Create a combined "positive" flag for Transfer or Appt Set
focus_calls['is_positive'] = focus_calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])

# Group by attempt: No Answer %, total calls, positive wins count
wins_per_attempt = focus_calls.groupby('attempt').agg(
    total_calls=('Lead ID', 'count'),
    no_answer_count=('Outcome', lambda x: (x == 'No Answer').sum()),
    positive_wins=('is_positive', 'sum')
)

wins_per_attempt['no_answer_%'] = (wins_per_attempt['no_answer_count'] / wins_per_attempt['total_calls'] * 100).round(1)
wins_per_attempt['positive_%'] = (wins_per_attempt['positive_wins'] / wins_per_attempt['total_calls'] * 100).round(2)

# Limit to attempts 1–20 (same as before)
wins_per_attempt = wins_per_attempt[wins_per_attempt.index <= 20]

# Reorder columns for clarity
wins_per_attempt = wins_per_attempt[['total_calls', 'no_answer_%', 'positive_wins', 'positive_%']]

print("=== No Answer % and Positive Wins (Transfer Sent + Appt Set) per Attempt — Movoto + DBP ===")
print("Attempt | Total Calls | No Answer % | Wins Count | Wins %")
print(wins_per_attempt.to_string())

=== No Answer % and Positive Wins (Transfer Sent + Appt Set) per Attempt — Movoto + DBP ===
Attempt | Total Calls | No Answer % | Wins Count | Wins %
         total_calls  no_answer_%  positive_wins  positive_%
attempt                                                     
0              31881         75.3           1378        4.32
1              17884         61.3           1479        8.27
2              13875         87.7            399        2.88
3              13220         92.8            268        2.03
4              13583         94.1            197        1.45
5              13127         94.9            155        1.18
6              12404         95.8            112        0.90
7              11758         96.5             92        0.78
8              11221         96.8             78        0.70
9              10796         97.0             66        0.61
10             10297         97.2             46        0.45
11              9924         97.1             52        0

In [59]:
# Assuming 'funnel_by_segment' is still in memory from your previous run
# If not, re-run the aggregation code first

# Add cumulative counts (running total from left to right)
funnel_by_segment['cum_contacted'] = funnel_by_segment['contacted_leads'].cumsum()
funnel_by_segment['cum_opp']       = funnel_by_segment['opportunities'].cumsum()
funnel_by_segment['cum_file']      = funnel_by_segment['file_started'].cumsum()
funnel_by_segment['cum_funded']    = funnel_by_segment['funded'].cumsum()

# Add drop-off counts (leads lost at each stage)
funnel_by_segment['lost_to_contact']   = funnel_by_segment['total_leads'] - funnel_by_segment['contacted_leads']
funnel_by_segment['lost_to_opp']       = funnel_by_segment['contacted_leads'] - funnel_by_segment['opportunities']
funnel_by_segment['lost_to_file']      = funnel_by_segment['opportunities'] - funnel_by_segment['file_started']
funnel_by_segment['lost_to_funded']    = funnel_by_segment['file_started'] - funnel_by_segment['funded']

print("=== Updated Funnel with Counts + Cumulative + Drop-off ===")
print(funnel_by_segment[[
    'total_leads', 'contacted_leads', 'lost_to_contact',
    'opportunities', 'lost_to_opp',
    'file_started', 'lost_to_file',
    'funded', 'lost_to_funded',
    'cum_contacted', 'cum_opp', 'cum_file', 'cum_funded'
]].sort_values('total_leads', ascending=False).to_string())

=== Updated Funnel with Counts + Cumulative + Drop-off ===
                      total_leads  contacted_leads  lost_to_contact  opportunities  lost_to_opp  file_started  lost_to_file  funded  lost_to_funded  cum_contacted  cum_opp  cum_file  cum_funded
Lead Channel Segment                                                                                                                                                                             
Digital Buy Partner         12509             5386             7123           2483         2903           762          1721      15             747           5386     2483       762          15
Movoto                       9236             6638             2598           2445         4193           605          1840       7             598          14072     6380      2338         107
Paid Spend                   5138             2934             2204           2275          659          1134          1141      73            1061          17080   

In [60]:
# Recompute funnel (no cumsum across segments)
funnel = leads.groupby('Lead Channel Segment').agg(
    Total_Leads=('Lead ID', 'nunique'),
    Contacted=('was_contacted', 'sum'),
    Opportunities=('had_opportunity', 'sum'),
    File_Started=('file_started', 'sum'),
    Funded=('funded', 'sum')
)

# Add rates (stage-to-stage)
funnel['Contact_%'] = (funnel['Contacted'] / funnel['Total_Leads'] * 100).round(1)
funnel['Opp_from_Contact_%'] = (funnel['Opportunities'] / funnel['Contacted'] * 100).round(1)
funnel['File_from_Opp_%'] = (funnel['File_Started'] / funnel['Opportunities'] * 100).round(1)
funnel['Funded_from_File_%'] = (funnel['Funded'] / funnel['File_Started'] * 100).round(1)

# Add drop-off counts
funnel['Lost_before_Contact'] = funnel['Total_Leads'] - funnel['Contacted']
funnel['Lost_before_Opp'] = funnel['Contacted'] - funnel['Opportunities']
funnel['Lost_before_File'] = funnel['Opportunities'] - funnel['File_Started']
funnel['Lost_before_Funded'] = funnel['File_Started'] - funnel['Funded']

# Reorder columns for funnel flow
funnel = funnel[[
    'Total_Leads', 'Lost_before_Contact', 'Contacted',
    'Lost_before_Opp', 'Opportunities',
    'Lost_before_File', 'File_Started',
    'Lost_before_Funded', 'Funded',
    'Contact_%', 'Opp_from_Contact_%', 'File_from_Opp_%', 'Funded_from_File_%'
]]

# Sort by Total_Leads descending (or change to 'Funded' if you prefer)
funnel = funnel.sort_values('Total_Leads', ascending=False)

print("=== Funnel per Segment: Counts + Drop-off + Stage-to-Stage Rates ===")
print(funnel.to_string(na_rep='—'))

=== Funnel per Segment: Counts + Drop-off + Stage-to-Stage Rates ===
                      Total_Leads  Lost_before_Contact  Contacted  Lost_before_Opp  Opportunities  Lost_before_File  File_Started  Lost_before_Funded  Funded  Contact_%  Opp_from_Contact_%  File_from_Opp_%  Funded_from_File_%
Lead Channel Segment                                                                                                                                                                                                             
Digital Buy Partner         12509                 7123       5386             2903           2483              1721           762                 747      15       43.1                46.1             30.7                 2.0
Movoto                       9236                 2598       6638             4193           2445              1840           605                 598       7       71.9                36.8             24.7                 1.2
Paid Spend                 

In [61]:
# Filter to just Movoto + Digital Buy Partner
movoto_dbp = leads[
    leads['Lead Channel Segment'].isin(['Movoto', 'Digital Buy Partner'])
].copy()

print("Total leads in Movoto + DBP:", len(movoto_dbp))
print("Unique Lead IDs:", movoto_dbp['Lead ID'].nunique())

Total leads in Movoto + DBP: 21745
Unique Lead IDs: 21745


In [65]:
# Aggregate funnel counts for Movoto + DBP only
movoto_dbp_funnel = movoto_dbp.agg({
    'Lead ID': 'nunique',
    'was_contacted': 'sum',
    'had_opportunity': 'sum',
    'file_started': 'sum',
    'funded': 'sum'
})

# Rename index to column names (agg returns Series with index as column names)
movoto_dbp_funnel = movoto_dbp_funnel.rename(index={
    'Lead ID': 'Total_Leads',
    'was_contacted': 'Contacted',
    'had_opportunity': 'Opportunities',
    'file_started': 'File_Started',
    'funded': 'Funded'
})

# Transpose to make it one row (now it's a DataFrame with 1 row)
movoto_dbp_funnel = movoto_dbp_funnel.to_frame().T

# Add stage-to-stage rates
movoto_dbp_funnel['Contact_%']     = (movoto_dbp_funnel['Contacted'] / movoto_dbp_funnel['Total_Leads'] * 100).round(1)
movoto_dbp_funnel['Opp_from_Contact_%'] = (movoto_dbp_funnel['Opportunities'] / movoto_dbp_funnel['Contacted'] * 100).round(1)
movoto_dbp_funnel['File_from_Opp_%'] = (movoto_dbp_funnel['File_Started'] / movoto_dbp_funnel['Opportunities'] * 100).round(1)
movoto_dbp_funnel['Funded_from_File_%'] = (movoto_dbp_funnel['Funded'] / movoto_dbp_funnel['File_Started'] * 100).round(1)

# Add drop-off counts
movoto_dbp_funnel['Lost_before_Contact'] = movoto_dbp_funnel['Total_Leads'] - movoto_dbp_funnel['Contacted']
movoto_dbp_funnel['Lost_before_Opp']     = movoto_dbp_funnel['Contacted'] - movoto_dbp_funnel['Opportunities']
movoto_dbp_funnel['Lost_before_File']    = movoto_dbp_funnel['Opportunities'] - movoto_dbp_funnel['File_Started']
movoto_dbp_funnel['Lost_before_Funded']  = movoto_dbp_funnel['File_Started'] - movoto_dbp_funnel['Funded']

print("=== Funnel: Movoto + Digital Buy Partner Combined ===")
print("Stage counts + drop-off + conversion rates")
print(movoto_dbp_funnel.to_string(index=False))

=== Funnel: Movoto + Digital Buy Partner Combined ===
Stage counts + drop-off + conversion rates
 Total_Leads  Contacted  Opportunities  File_Started  Funded  Contact_%  Opp_from_Contact_%  File_from_Opp_%  Funded_from_File_%  Lost_before_Contact  Lost_before_Opp  Lost_before_File  Lost_before_Funded
       21745      12024           4928          1367      22       55.3                41.0             27.7                 1.6                 9721             7096              3561                1345


In [66]:
# Aggregate funnel counts for Movoto + DBP only
movoto_dbp_funnel = movoto_dbp.agg({
    'Lead ID': 'nunique',
    'was_contacted': 'sum',
    'had_opportunity': 'sum',
    'file_started': 'sum',
    'funded': 'sum'
})

# Rename columns for clarity
movoto_dbp_funnel = movoto_dbp_funnel.rename(index={
    'Lead ID': 'Total_Leads',
    'was_contacted': 'Contacted',
    'had_opportunity': 'Opportunities',
    'file_started': 'File_Started',
    'funded': 'Funded'
}).to_frame().T  # now transpose to make it one row

# Add stage-to-stage rates
movoto_dbp_funnel['Contact_%']     = (movoto_dbp_funnel['Contacted'] / movoto_dbp_funnel['Total_Leads'] * 100).round(1)
movoto_dbp_funnel['Opp_from_Contact_%'] = (movoto_dbp_funnel['Opportunities'] / movoto_dbp_funnel['Contacted'] * 100).round(1)
movoto_dbp_funnel['File_from_Opp_%'] = (movoto_dbp_funnel['File_Started'] / movoto_dbp_funnel['Opportunities'] * 100).round(1)
movoto_dbp_funnel['Funded_from_File_%'] = (movoto_dbp_funnel['Funded'] / movoto_dbp_funnel['File_Started'] * 100).round(1)

# Add drop-off counts
movoto_dbp_funnel['Lost_before_Contact'] = movoto_dbp_funnel['Total_Leads'] - movoto_dbp_funnel['Contacted']
movoto_dbp_funnel['Lost_before_Opp']     = movoto_dbp_funnel['Contacted'] - movoto_dbp_funnel['Opportunities']
movoto_dbp_funnel['Lost_before_File']    = movoto_dbp_funnel['Opportunities'] - movoto_dbp_funnel['File_Started']
movoto_dbp_funnel['Lost_before_Funded']  = movoto_dbp_funnel['File_Started'] - movoto_dbp_funnel['Funded']

print("=== Funnel: Movoto + Digital Buy Partner Combined ===")
print("Stage counts + drop-off + conversion rates")
print(movoto_dbp_funnel.to_string())

=== Funnel: Movoto + Digital Buy Partner Combined ===
Stage counts + drop-off + conversion rates
   Total_Leads  Contacted  Opportunities  File_Started  Funded  Contact_%  Opp_from_Contact_%  File_from_Opp_%  Funded_from_File_%  Lost_before_Contact  Lost_before_Opp  Lost_before_File  Lost_before_Funded
0        21745      12024           4928          1367      22       55.3                41.0             27.7                 1.6                 9721             7096              3561                1345


In [67]:
# Create flag for leads with at least one Transfer Sent or Appointment Set
leads_with_win = leads[leads['Lead ID'].isin(
    calls[calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])]['Lead ID'].unique()
)].copy()

print("Number of leads with at least one Transfer Sent or Appointment Set:", len(leads_with_win))
print("\nBreakdown by segment:")
print(leads_with_win['Lead Channel Segment'].value_counts(dropna=False))

Number of leads with at least one Transfer Sent or Appointment Set: 6123

Breakdown by segment:
Lead Channel Segment
Movoto                 2220
Digital Buy Partner    1727
Paid Spend             1355
Direct / Organic        684
Retargeting              90
Other                    42
Mail                      5
Name: count, dtype: int64


In [68]:
# The warm-win leads are already identified in your previous step
# We filter the leads table to only those 6,123 IDs
warm_win_leads = leads[leads['Lead ID'].isin(
    calls[calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])]['Lead ID'].unique()
)].copy()

# Aggregate File Started count and rate per segment
file_rate_table = warm_win_leads.groupby('Lead Channel Segment').agg(
    warm_wins=('Lead ID', 'nunique'),
    file_started_count=('file_started', 'sum')
)

file_rate_table['file_started_rate_%'] = (file_rate_table['file_started_count'] / file_rate_table['warm_wins'] * 100).round(1)

# Sort by file_started_rate descending
file_rate_table = file_rate_table.sort_values('file_started_rate_%', ascending=False)

print("=== File Started Rate after Warm Win (Transfer Sent or Appointment Set) ===")
print("Per segment: # warm-win leads | # reached File Started | % rate")
print(file_rate_table.to_string())

=== File Started Rate after Warm Win (Transfer Sent or Appointment Set) ===
Per segment: # warm-win leads | # reached File Started | % rate
                      warm_wins  file_started_count  file_started_rate_%
Lead Channel Segment                                                    
Mail                          5                   4                 80.0
Other                        42                  25                 59.5
Direct / Organic            684                 383                 56.0
Paid Spend                 1355                 665                 49.1
Digital Buy Partner        1727                 510                 29.5
Retargeting                  90                  22                 24.4
Movoto                     2220                 382                 17.2


In [69]:
# Get the warm handoff calls for Movoto + DBP only
warm_handoff_calls = calls[
    (calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])) &
    (calls['Lead ID'].isin(movoto_dbp['Lead ID']))
].copy()

# Add segment
warm_handoff_calls = warm_handoff_calls.merge(
    leads[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='left'
)

# Outcome distribution per segment
outcome_dist = warm_handoff_calls.groupby(['Lead Channel Segment', 'Outcome']).size().unstack(fill_value=0)

print("=== Warm Handoff Outcomes (Transfer Sent / Appointment Set) in Movoto + DBP ===")
print(outcome_dist.to_string())

=== Warm Handoff Outcomes (Transfer Sent / Appointment Set) in Movoto + DBP ===
Outcome               Appointment Set  Transfer Sent
Lead Channel Segment                                
Digital Buy Partner               509           1378
Movoto                           1458           1193


In [70]:
# Get unique Lead IDs with at least one Transfer Sent or Appointment Set
warm_handoff_leads = calls[
    calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])
]['Lead ID'].unique()

# Filter the leads table to only these leads
warm_handoff_leads_df = leads[leads['Lead ID'].isin(warm_handoff_leads)].copy()

# Quick sanity checks
print("Number of leads with at least one warm handoff:", len(warm_handoff_leads_df))
print("\nBreakdown by Lead Channel Segment (top 5):")
print(warm_handoff_leads_df['Lead Channel Segment'].value_counts().head())

print("\nUnique Lead IDs:", warm_handoff_leads_df['Lead ID'].nunique())

Number of leads with at least one warm handoff: 6123

Breakdown by Lead Channel Segment (top 5):
Lead Channel Segment
Movoto                 2220
Digital Buy Partner    1727
Paid Spend             1355
Direct / Organic        684
Retargeting              90
Name: count, dtype: int64

Unique Lead IDs: 6123


In [71]:
# Filter leads to only those with warm handoff
warm_win_leads_df = leads[leads['Lead ID'].isin(warm_handoff_leads)].copy()

# Aggregate File Started rate per segment
file_started_table = warm_win_leads_df.groupby('Lead Channel Segment').agg(
    warm_win_leads=('Lead ID', 'nunique'),
    file_started_count=('file_started', 'sum')
)

file_started_table['file_started_rate_%'] = (file_started_table['file_started_count'] / file_started_table['warm_win_leads'] * 100).round(1)

# Sort by rate descending
file_started_table = file_started_table.sort_values('file_started_rate_%', ascending=False)

print("=== File Started Rate after Warm Handoff (Transfer Sent or Appointment Set) ===")
print("Per segment: # warm-win leads | # reached File Started | % rate")
print(file_started_table.to_string())

=== File Started Rate after Warm Handoff (Transfer Sent or Appointment Set) ===
Per segment: # warm-win leads | # reached File Started | % rate
                      warm_win_leads  file_started_count  file_started_rate_%
Lead Channel Segment                                                         
Mail                               5                   4                 80.0
Other                             42                  25                 59.5
Direct / Organic                 684                 383                 56.0
Paid Spend                      1355                 665                 49.1
Digital Buy Partner             1727                 510                 29.5
Retargeting                       90                  22                 24.4
Movoto                          2220                 382                 17.2


In [72]:
# Filter leads to Movoto + Digital Buy Partner only
movoto_dbp_leads = leads[
    leads['Lead Channel Segment'].isin(['Movoto', 'Digital Buy Partner'])
].copy()

print("Total leads in Movoto + DBP:", len(movoto_dbp_leads))
print("Unique Lead IDs:", movoto_dbp_leads['Lead ID'].nunique())

# Get Lead IDs with at least one Transfer Sent or Appointment Set
warm_handoff_ids = calls[
    calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])
]['Lead ID'].unique()

# Filter movoto_dbp_leads to only those with warm handoff AND opportunity
movoto_dbp_warm_opp = movoto_dbp_leads[
    (movoto_dbp_leads['Lead ID'].isin(warm_handoff_ids)) &
    (movoto_dbp_leads['had_opportunity'] == True)
].copy()

print("Number of Movoto + DBP leads with warm handoff AND opportunity:", len(movoto_dbp_warm_opp))
print("\nBreakdown by segment:")
print(movoto_dbp_warm_opp['Lead Channel Segment'].value_counts())

Total leads in Movoto + DBP: 21745
Unique Lead IDs: 21745
Number of Movoto + DBP leads with warm handoff AND opportunity: 2952

Breakdown by segment:
Lead Channel Segment
Digital Buy Partner    1494
Movoto                 1458
Name: count, dtype: int64


In [73]:
# The list of lead IDs with warm handoff + opportunity
warm_opp_ids = movoto_dbp_warm_opp['Lead ID'].unique()

# Get all calls for these leads
calls_for_warm_opp = calls[calls['Lead ID'].isin(warm_opp_ids)].copy()

# Add segment for grouping
calls_for_warm_opp = calls_for_warm_opp.merge(
    movoto_dbp_warm_opp[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='left'
)

# Define loan officer roles
loan_officer_roles = [
    'Loan Advisor',
    'Loan Prospector',
    'Loan Officer Assistant',
    'Sr Loan Advisor'
]

# Filter to calls by loan officer roles
loan_officer_calls = calls_for_warm_opp[
    calls_for_warm_opp['Team Member Role'].isin(loan_officer_roles)
].copy()

print("Total calls by loan officer roles for these leads:", len(loan_officer_calls))
print("\nCalls per segment:")
print(loan_officer_calls['Lead Channel Segment'].value_counts(dropna=False))

print("\nOutcome distribution (top 10):")
print(loan_officer_calls['Outcome'].value_counts(dropna=False).head(10))

Total calls by loan officer roles for these leads: 2720

Calls per segment:
Lead Channel Segment
Movoto                 1579
Digital Buy Partner    1141
Name: count, dtype: int64

Outcome distribution (top 10):
Outcome
No Answer                   1174
Left Voicemail               364
Hung Up                      185
Called - Follow Up           167
Appointment Set              144
Application Completed        141
Pitch Completed              115
Transfer Sent                 86
Agent - Called Follow Up      62
Agent - Pitch Completed       42
Name: count, dtype: int64


In [74]:
# Add file_started flag from leads (join if not already present)
loan_officer_calls = loan_officer_calls.merge(
    leads[['Lead ID', 'file_started']],
    on='Lead ID',
    how='left'
)

# Separate calls into two groups
calls_file_started = loan_officer_calls[loan_officer_calls['file_started'] == True]
calls_no_file = loan_officer_calls[loan_officer_calls['file_started'] == False]

# Outcome distribution for each group
print("=== Outcome distribution in post-handoff calls by loan officer roles ===")
print("\nCalls where lead reached File Started (count):", len(calls_file_started))
print(calls_file_started['Outcome'].value_counts().head(10))

print("\nCalls where lead did NOT reach File Started (count):", len(calls_no_file))
print(calls_no_file['Outcome'].value_counts().head(10))

=== Outcome distribution in post-handoff calls by loan officer roles ===

Calls where lead reached File Started (count): 1324
Outcome
No Answer                   406
Left Voicemail              207
Called - Follow Up          142
Application Completed       111
Pitch Completed             111
Hung Up                      72
Appointment Set              58
Agent - Called Follow Up     49
Agent - Pitch Completed      40
Transfer Sent                25
Name: count, dtype: int64

Calls where lead did NOT reach File Started (count): 1396
Outcome
No Answer                768
Left Voicemail           157
Hung Up                  113
Appointment Set           86
Transfer Sent             61
Application Completed     30
Redial                    25
Called - Follow Up        25
Application Started       22
Nurture - No Benefit      17
Name: count, dtype: int64


In [75]:
# Assuming loan_officer_calls is still in memory from the previous step
# If not, re-run the earlier block that created it

# Pivot: Outcome vs file_started (Yes/No)
pivot_outcomes = pd.crosstab(
    loan_officer_calls['Outcome'],
    loan_officer_calls['file_started'],
    margins=True,
    margins_name='Total'
)

# Rename columns for clarity
pivot_outcomes.columns = ['No File Started', 'File Started', 'Total']

# Add % for each group
pivot_outcomes['File Started %'] = (pivot_outcomes['File Started'] / pivot_outcomes['Total'] * 100).round(1)
pivot_outcomes['No File Started %'] = (pivot_outcomes['No File Started'] / pivot_outcomes['Total'] * 100).round(1)

# Add difference column
pivot_outcomes['Difference (File - No File) %'] = (pivot_outcomes['File Started %'] - pivot_outcomes['No File Started %']).round(1)

# Sort by Total descending
pivot_outcomes = pivot_outcomes.sort_values('Total', ascending=False)

print("=== Pivot: Post-handoff Calls by Outcome & File Started Status ===")
print("Rows: Outcome | Columns: No File Started / File Started / Total | Rates & Difference")
print(pivot_outcomes.to_string())

=== Pivot: Post-handoff Calls by Outcome & File Started Status ===
Rows: Outcome | Columns: No File Started / File Started / Total | Rates & Difference
                                                               No File Started  File Started  Total  File Started %  No File Started %  Difference (File - No File) %
Outcome                                                                                                                                                              
Total                                                                     1394          1310   2704            48.4               51.6                           -3.2
No Answer                                                                  768           406   1174            34.6               65.4                          -30.8
Left Voicemail                                                             157           207    364            56.9               43.1                           13.8
Hung Up           

In [76]:
# Assuming 'pivot_outcomes' is still in memory from your last run
# If not, re-run the crosstab code first to recreate it

# Calculate % within each group (column-wise)
pivot_outcomes['File Started % of Group'] = (pivot_outcomes['File Started'] / pivot_outcomes['File Started']['Total'] * 100).round(1)
pivot_outcomes['No File Started % of Group'] = (pivot_outcomes['No File Started'] / pivot_outcomes['No File Started']['Total'] * 100).round(1)

# Keep the Difference column for comparison
pivot_outcomes['Difference (File - No File) %'] = (pivot_outcomes['File Started % of Group'] - pivot_outcomes['No File Started % of Group']).round(1)

# Sort by Total descending again
pivot_outcomes = pivot_outcomes.sort_values('Total', ascending=False)

print("=== Updated Pivot: Post-handoff Calls by Outcome & File Started ===")
print("Now with % of total calls in each group (File Started group vs No File Started group)")
print(pivot_outcomes[['No File Started', 'File Started', 'Total', 'No File Started % of Group', 'File Started % of Group', 'Difference (File - No File) %']].to_string())

=== Updated Pivot: Post-handoff Calls by Outcome & File Started ===
Now with % of total calls in each group (File Started group vs No File Started group)
                                                               No File Started  File Started  Total  No File Started % of Group  File Started % of Group  Difference (File - No File) %
Outcome                                                                                                                                                                                
Total                                                                     1394          1310   2704                       100.0                    100.0                            0.0
No Answer                                                                  768           406   1174                        55.1                     31.0                          -24.1
Left Voicemail                                                             157           207    364           

In [77]:
# Filter calls to Movoto + DBP
focus_segments = ['Movoto', 'Digital Buy Partner']
focus_calls = calls_with_segment[calls_with_segment['Lead Channel Segment'].isin(focus_segments)].copy()

# Ensure attempt is integer
focus_calls['attempt'] = focus_calls['Pool A Call Attempt Number'].astype(int, errors='ignore').fillna(0).astype(int)

# Flag positive outcomes
focus_calls['is_positive'] = focus_calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])

# Aggregate per attempt
wins_table = focus_calls.groupby('attempt').agg(
    total_calls=('Lead ID', 'count'),
    positive_wins=('is_positive', 'sum')
)

wins_table['positive_%'] = (wins_table['positive_wins'] / wins_table['total_calls'] * 100).round(2)

# Cumulative wins
wins_table = wins_table.sort_index()
wins_table['cumulative_wins'] = wins_table['positive_wins'].cumsum()
wins_table['cumulative_%_of_all_wins'] = (wins_table['cumulative_wins'] / wins_table['positive_wins'].sum() * 100).round(1)

# Limit to attempt 1–20 + show only relevant columns
wins_table = wins_table[wins_table.index <= 20][['total_calls', 'positive_wins', 'positive_%', 'cumulative_wins', 'cumulative_%_of_all_wins']]

print("=== Positive Wins (Transfer + Appt Set) & Cumulative % by Attempt — Movoto + DBP ===")
print(wins_table.to_string())

=== Positive Wins (Transfer + Appt Set) & Cumulative % by Attempt — Movoto + DBP ===
         total_calls  positive_wins  positive_%  cumulative_wins  cumulative_%_of_all_wins
attempt                                                                                   
0              31881           1378        4.32             1378                      30.4
1              17884           1479        8.27             2857                      63.0
2              13875            399        2.88             3256                      71.7
3              13220            268        2.03             3524                      77.7
4              13583            197        1.45             3721                      82.0
5              13127            155        1.18             3876                      85.4
6              12404            112        0.90             3988                      87.9
7              11758             92        0.78             4080                      89.9
8    

In [78]:
# Leads with at least one Transfer Sent or Appointment Set
warm_win_leads = leads[leads['Lead ID'].isin(
    calls[calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])]['Lead ID'].unique()
)].copy()

# Aggregate File Started rate per segment
file_rate_table = warm_win_leads.groupby('Lead Channel Segment').agg(
    warm_win_leads=('Lead ID', 'nunique'),
    file_started_count=('file_started', 'sum')
)

file_rate_table['file_started_rate_%'] = (file_rate_table['file_started_count'] / file_rate_table['warm_win_leads'] * 100).round(1)

file_rate_table = file_rate_table.sort_values('file_started_rate_%', ascending=False)

print("=== File Started Rate after Warm Handoff (Transfer Sent or Appointment Set) ===")
print(file_rate_table.to_string())

=== File Started Rate after Warm Handoff (Transfer Sent or Appointment Set) ===
                      warm_win_leads  file_started_count  file_started_rate_%
Lead Channel Segment                                                         
Mail                               5                   4                 80.0
Other                             42                  25                 59.5
Direct / Organic                 684                 383                 56.0
Paid Spend                      1355                 665                 49.1
Digital Buy Partner             1727                 510                 29.5
Retargeting                       90                  22                 24.4
Movoto                          2220                 382                 17.2


In [79]:
# Assuming loan_officer_calls already exists from earlier (post-handoff calls by loan officer roles)
# If not, re-run the earlier block that created loan_officer_calls

# Pivot: Outcome vs file_started
pivot_drivers = pd.crosstab(
    loan_officer_calls['Outcome'],
    loan_officer_calls['file_started'],
    margins=True,
    margins_name='Total'
)

pivot_drivers.columns = ['No File Started', 'File Started', 'Total']

# Add % within each group
pivot_drivers['File Started % of Group'] = (pivot_drivers['File Started'] / pivot_drivers.loc['Total', 'File Started'] * 100).round(1)
pivot_drivers['No File Started % of Group'] = (pivot_drivers['No File Started'] / pivot_drivers.loc['Total', 'No File Started'] * 100).round(1)

# Difference column
pivot_drivers['Difference (File - No File) %'] = (pivot_drivers['File Started % of Group'] - pivot_drivers['No File Started % of Group']).round(1)

# Sort by Total descending
pivot_drivers = pivot_drivers.sort_values('Total', ascending=False)

# Optional: filter to top outcomes only (cleaner table)
top_outcomes = pivot_drivers['Total'].sort_values(ascending=False).head(12).index.tolist()
pivot_drivers_top = pivot_drivers.loc[top_outcomes]

print("=== Post-handoff Outcomes by File Started Status (Loan Officer Roles Only) ===")
print("Top outcomes + % of group + difference")
print(pivot_drivers_top.to_string())

=== Post-handoff Outcomes by File Started Status (Loan Officer Roles Only) ===
Top outcomes + % of group + difference
                          No File Started  File Started  Total  File Started % of Group  No File Started % of Group  Difference (File - No File) %
Outcome                                                                                                                                           
Total                                1394          1310   2704                    100.0                       100.0                            0.0
No Answer                             768           406   1174                     31.0                        55.1                          -24.1
Left Voicemail                        157           207    364                     15.8                        11.3                            4.5
Hung Up                               113            72    185                      5.5                         8.1                           -2.6


In [80]:
# Create the file-started flag if not already there
leads['file_started'] = leads['Number of File Started Leads'] >= 1

# Filter to leads with file started
file_started_leads = leads[leads['file_started'] == True].copy()

print("Total leads with File Started >= 1:", len(file_started_leads))
print("\nCount per segment:")
print(file_started_leads['Lead Channel Segment'].value_counts())

# Filter to the file-started leads we just confirmed
file_started_leads = leads[leads['file_started'] == True].copy()

# Aggregate downstream stages per segment
downstream_funnel = file_started_leads.groupby('Lead Channel Segment').agg(
    File_Started=('file_started', 'sum'),
    Entered_Processing=('Number of Entered Processing Leads', lambda x: (x >= 1).sum()),
    Pre_Approval=('Number of Pre-Approval Leads', lambda x: (x >= 1).sum()),
    Funded=('funded', 'sum')
)

# Add stage-to-stage rates
downstream_funnel['Processing_from_File_%'] = (downstream_funnel['Entered_Processing'] / downstream_funnel['File_Started'] * 100).round(1)
downstream_funnel['PreApproval_from_Processing_%'] = (downstream_funnel['Pre_Approval'] / downstream_funnel['Entered_Processing'] * 100).round(1)
downstream_funnel['Funded_from_PreApproval_%'] = (downstream_funnel['Funded'] / downstream_funnel['Pre_Approval'] * 100).round(1)

# Reorder columns for funnel flow
downstream_funnel = downstream_funnel[[
    'File_Started', 'Entered_Processing', 'Pre_Approval', 'Funded',
    'Processing_from_File_%', 'PreApproval_from_Processing_%', 'Funded_from_PreApproval_%'
]]

print("=== Downstream Funnel: After File Started (per segment) ===")
print("Counts + stage-to-stage conversion rates")
print(downstream_funnel.to_string(na_rep='—'))

Total leads with File Started >= 1: 3642

Count per segment:
Lead Channel Segment
Paid Spend             1134
Direct / Organic        914
Digital Buy Partner     762
Movoto                  605
Retargeting             136
Mail                     57
Other                    34
Name: count, dtype: int64
=== Downstream Funnel: After File Started (per segment) ===
Counts + stage-to-stage conversion rates
                      File_Started  Entered_Processing  Pre_Approval  Funded  Processing_from_File_%  PreApproval_from_Processing_%  Funded_from_PreApproval_%
Lead Channel Segment                                                                                                                                          
Digital Buy Partner            762                  63             3      15                     8.3                            4.8                      500.0
Direct / Organic               914                 189           177      84                    20.7                  

In [81]:
print("Unique Current Lead Status values containing 'Nurture' (case-insensitive):")
print(leads[leads['Current Lead Status'].str.contains('nurture', case=False, na=False)]['Current Lead Status'].value_counts())

print("\nTotal leads with any nurture-related status:", 
      leads['Current Lead Status'].str.contains('nurture', case=False, na=False).sum())



Unique Current Lead Status values containing 'Nurture' (case-insensitive):
Current Lead Status
Nurture    8289
Name: count, dtype: int64

Total leads with any nurture-related status: 8289


In [82]:
# Filter leads to only those with Current Lead Status == 'Nurture'
nurture_leads = leads[leads['Current Lead Status'] == 'Nurture'].copy()

# Aggregate count per segment
nurture_by_segment = nurture_leads.groupby('Lead Channel Segment').agg(
    nurture_leads=('Lead ID', 'nunique')
).sort_values('nurture_leads', ascending=False)

# Add % of total nurture leads
nurture_by_segment['%_of_total_nurture'] = (nurture_by_segment['nurture_leads'] / len(nurture_leads) * 100).round(1)

print("=== Leads with Current Lead Status = 'Nurture' — Breakdown by Segment ===")
print(nurture_by_segment.to_string())

=== Leads with Current Lead Status = 'Nurture' — Breakdown by Segment ===
                      nurture_leads  %_of_total_nurture
Lead Channel Segment                                   
Movoto                         3696                44.6
Digital Buy Partner            2145                25.9
Paid Spend                     1350                16.3
Direct / Organic                882                10.6
Mail                            104                 1.3
Retargeting                      89                 1.1
Other                            23                 0.3


In [84]:
# Filter to Movoto + DBP
movoto_dbp = leads[leads['Lead Channel Segment'].isin(['Movoto', 'Digital Buy Partner'])].copy()

# Create warm handoff flag
warm_handoff_ids = calls[calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])]['Lead ID'].unique()
movoto_dbp['had_warm_handoff'] = movoto_dbp['Lead ID'].isin(warm_handoff_ids)

# Aggregate with named columns (agg returns a DataFrame with one row)
baseline = movoto_dbp.agg({
    'Lead ID': 'nunique',
    'was_contacted': 'sum',
    'had_warm_handoff': 'sum'
})

# Rename columns cleanly
baseline = baseline.rename({
    'Lead ID': 'total_leads',
    'was_contacted': 'contacted',
    'had_warm_handoff': 'warm_handoff'
}).to_frame().T  # now transpose to make it a 1-row DataFrame

# Calculate rates
baseline['contact_rate_%']              = (baseline['contacted'] / baseline['total_leads'] * 100).round(1)
baseline['warm_handoff_from_contact_%'] = (baseline['warm_handoff'] / baseline['contacted'] * 100).round(1)
baseline['warm_handoff_from_total_%']   = (baseline['warm_handoff'] / baseline['total_leads'] * 100).round(1)

print("=== Current Baseline — Movoto + Digital Buy Partner Combined ===")
print(baseline.to_string(index=False))

=== Current Baseline — Movoto + Digital Buy Partner Combined ===
 total_leads  contacted  warm_handoff  contact_rate_%  warm_handoff_from_contact_%  warm_handoff_from_total_%
       21745      12024          3947            55.3                         32.8                       18.2


In [85]:
# Step 1: Get all leads with at least one Transfer Sent or Appointment Set
warm_handoff_leads = calls[calls['Outcome'].isin(['Transfer Sent', 'Appointment Set'])]['Lead ID'].unique()

# Step 2: Filter leads table to only these leads
warm_handoff_df = leads[leads['Lead ID'].isin(warm_handoff_leads)].copy()

# Step 3: Join their full call history (Pool A outbound)
warm_handoff_calls = calls[calls['Lead ID'].isin(warm_handoff_leads)].copy()

# Add segment info
warm_handoff_calls = warm_handoff_calls.merge(
    leads[['Lead ID', 'Lead Channel Segment']],
    on='Lead ID',
    how='left'
)

# Step 4: Quick sanity check
print("Number of leads with warm handoff:", len(warm_handoff_df))
print("Number of calls for these leads:", len(warm_handoff_calls))
print("\nCalls per segment:")
print(warm_handoff_calls['Lead Channel Segment'].value_counts(dropna=False))

Number of leads with warm handoff: 6123
Number of calls for these leads: 35567

Calls per segment:
Lead Channel Segment
Movoto                 15656
Digital Buy Partner    10058
Paid Spend              6373
Direct / Organic        3021
Retargeting              245
Other                    191
Mail                      13
NaN                       10
Name: count, dtype: int64


In [86]:
# Step 1: Add attempt bin to the warm-handoff calls
warm_handoff_calls['attempt'] = warm_handoff_calls['Pool A Call Attempt Number'].astype(int, errors='ignore').fillna(0).astype(int)

# Define the same bin function (copy from earlier if not defined)
def bin_attempt(n):
    if n == 1:
        return 'Attempt 1'
    elif 1 <= n <= 3:
        return 'Attempts 1–3'
    elif 4 <= n <= 6:
        return 'Attempts 4–6'
    elif 7 <= n <= 9:
        return 'Attempts 7–9'
    elif 10 <= n <= 12:
        return 'Attempts 10–12'
    elif 13 <= n <= 15:
        return 'Attempts 13–15'
    else:
        return '16+'

warm_handoff_calls['attempt_bin'] = warm_handoff_calls['attempt'].apply(bin_attempt)

# Step 2: Get first success per lead (to assign the bin correctly)
first_success = warm_handoff_calls.sort_values(['Lead ID', 'attempt']).groupby('Lead ID').head(1)[['Lead ID', 'attempt_bin']]

# Step 3: Join first bin back to leads for rate calculation
warm_handoff_with_bin = warm_win_leads_df.merge(
    first_success,
    on='Lead ID',
    how='left'
)

# Step 4: Aggregate warm-handoff count and file started rate per bin per segment
funnel_table = warm_handoff_with_bin.groupby(['Lead Channel Segment', 'attempt_bin']).agg(
    warm_handoff_leads=('Lead ID', 'nunique'),
    file_started_count=('file_started', 'sum')
)

funnel_table['file_started_rate_%'] = (funnel_table['file_started_count'] / funnel_table['warm_handoff_leads'] * 100).round(1)

# Add warm-handoff rate (as % of total leads in bin, but since this is already warm-win leads, we can show count only or skip rate if not meaningful)
# Pivot for readability
pivot_funnel = funnel_table.pivot_table(
    index='Lead Channel Segment',
    columns='attempt_bin',
    values=['warm_handoff_leads', 'file_started_rate_%'],
    aggfunc='first'
)

# Flatten columns
pivot_funnel.columns = [f"{col[1]} ({col[0]})" for col in pivot_funnel.columns]

print("=== Warm-Handoff Leads & File Started Rate by Attempt Bin per Segment ===")
print("Only leads whose first success was in that bin")
print(pivot_funnel.to_string(na_rep='—'))

=== Warm-Handoff Leads & File Started Rate by Attempt Bin per Segment ===
Only leads whose first success was in that bin
                      16+ (file_started_rate_%)  Attempt 1 (file_started_rate_%)  Attempts 13–15 (file_started_rate_%)  Attempts 1–3 (file_started_rate_%)  Attempts 4–6 (file_started_rate_%)  Attempts 7–9 (file_started_rate_%)  16+ (warm_handoff_leads)  Attempt 1 (warm_handoff_leads)  Attempts 13–15 (warm_handoff_leads)  Attempts 1–3 (warm_handoff_leads)  Attempts 4–6 (warm_handoff_leads)  Attempts 7–9 (warm_handoff_leads)
Lead Channel Segment                                                                                                                                                                                                                                                                                                                                                                                                                      
Digital Buy Partner      